# Gemma 4-E4B 微调教程

本Notebook演示如何使用 **Unsloth** 框架微调 Google Gemma 4-E4B 模型。

## 特点

- 使用 QLoRA 4-bit量化，仅需约10GB VRAM
- 支持在免费Colab T4 GPU上运行
- 训练速度提升2x，VRAM节省60%

## 硬件需求

| 模型       | QLoRA VRAM | 推荐GPU             |
| ---------- | ---------- | ------------------- |
| E4B (4.5B) | ~10 GB     | Colab T4 / RTX 3060 |
| 31B        | ~22 GB     | RTX 4090            |

## 多GPU策略 (DistributedConfig统一配置)

⚠️ **Notebook模式无法实现DDP数据并行** (需要多进程torchrun)

| 环境                       | device_map | 训练模式      | DistributedConfig                                 |
| -------------------------- | ---------- | ------------- | ------------------------------------------------- |
| Notebook (单进程)          | `{"": 0}`  | 单GPU         | `mode='single_gpu'`                               |
| torchrun 8卡DDP            | 不设置     | 数据并行      | `mode='ddp', models_per_gpu=1`                    |
| torchrun 8卡DDP+2倍吞吐    | 不设置     | 数据并行      | `mode='ddp', models_per_gpu=2`                    |
| torchrun 8卡FSDP           | 不设置     | 分片并行      | `mode='fsdp'`                                     |
| torchrun device_map 2D并行 | balanced   | 模型+数据并行 | `mode='device_map', gpu_groups=[[0,1],[2,3],...]` |

⚠️ **`device_map="balanced"`会导致模型并行而非数据并行**: 模型被拆分到多卡, 仅1个进程, 无法实现8x加速

---


## 0. 环境安装

首先安装 Unsloth 及相关依赖库。


In [1]:
# 安装 Unsloth 和相关依赖
# 注意：在 Colab 上运行时，取消下面的注释

# %%capture
# import os
# if 'COLAB_' not in ''.join(os.environ.keys()):
#     !pip install unsloth
# else:
#     !pip install --no-deps bitsandbytes accelerate xformers==0.0.29 peft trl triton cut_cross_entropy unsloth_zoo
#     !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
#     !pip install --no-deps unsloth

In [2]:
# 本地环境安装（非Colab）
try:
    import unsloth

    print(f"Unsloth 已安装，版本: {unsloth.__version__}")
except ImportError:
    print("正在安装 Unsloth...")
    import subprocess

    subprocess.run(["pip", "install", "unsloth"], check=True)
    import unsloth

    print(f"Unsloth 安装完成，版本: {unsloth.__version__}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth 已安装，版本: 2026.4.6


In [3]:
# Imports
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

_nb_file = globals().get("__vsc_ipynb_file__", "")
_bootstrap_starts = [Path.cwd()] + ([Path(_nb_file).parent] if _nb_file else [])
for _start in _bootstrap_starts:
    for _candidate in [_start] + list(_start.parents):
        if (_candidate / "pyproject.toml").exists():
            if str(_candidate) not in sys.path:
                sys.path.insert(0, str(_candidate))
            break

from unsloth_finetune.notebooking.common import initialize_notebook_context

NOTEBOOK_CONTEXT = initialize_notebook_context(
    notebook_file=_nb_file,
    cwd=Path.cwd(),
    configure_unsloth_cache=True,
)
NOTEBOOK_DIR = NOTEBOOK_CONTEXT["NOTEBOOK_DIR"]
PROJECT_ROOT = NOTEBOOK_CONTEXT["PROJECT_ROOT"]
UNSLOTH_CACHE_DIR = NOTEBOOK_CONTEXT["UNSLOTH_CACHE_DIR"]

import torch
from datasets import load_dataset
from PIL import Image
from trl import SFTConfig, SFTTrainer
from unsloth import FastVisionModel

print(f"Unsloth compile cache: {UNSLOTH_CACHE_DIR}")

# Check GPU availability
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {gpu_memory:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("Warning: no GPU detected, training will be very slow!")

Unsloth compile cache: /raid5/sh/code/unsloth-finetune/notebooks/unsloth_compiled_cache
GPU: NVIDIA RTX 5880 Ada Generation
VRAM: 47.4 GB
CUDA version: 12.8


## 1. 配置参数

修改以下配置单元格中的参数即可适配不同环境和训练需求。所有路径基于 `PROJECT_ROOT` 自动推导，无需手动调整其他单元格。

**跨Notebook参数关联:**

- `DATA_PATH`: 由 **01-数据预处理** Notebook 生成的训练数据路径
- `LORA_OUTPUT_DIR`: 微调模型输出路径 (层级化: 模型名/模式/时间戳)，**03-推理** 和 **04-对比** Notebook 将引用此路径
- `LORA_OUTPUT_LATEST`: 最新训练结果的路径 (通过latest.txt自动定位)，辩Notebook引用时优先使用


In [4]:
# ============================================================
# 项目路径与全局配置
# ============================================================
# 【重要】修改以下参数即可适配不同环境，无需改动其他单元格

# ---------- 训练模式选择 ----------
# "single"      → 单GPU微调流程 (Section 2-9)
# "DDP"         → DDP 8卡分布式微调 (Section 10)
# "DDP" + MODELS_PER_GPU=2 → DDP 8卡 + 2倍吞吐 (小模型场景)
# "device_map"  → device_map 2D并行: 8卡分4组 (Section 10, 大模型场景)
# "FSDP"        → FSDP 8卡分布式微调 (Section 10)
# "auto"        → 自动检测模式 (根据MODEL_VRAM_GB自动选择最优模式)
# "multi_node"  → 多机多卡训练 (Section 10)
# "compare"     → 性能对比分析 (Section 10.5, 需先完成训练)
TRAINING_MODE = "DDP"

VALID_MODES = {"single", "DDP", "device_map", "FSDP", "auto", "multi_node", "compare"}
if TRAINING_MODE not in VALID_MODES:
    raise ValueError(f"TRAINING_MODE必须是{VALID_MODES}之一, 当前: {TRAINING_MODE}")

print(f"训练模式: {TRAINING_MODE}")

NOTEBOOK_DIR = NOTEBOOK_CONTEXT["NOTEBOOK_DIR"]
PROJECT_ROOT = NOTEBOOK_CONTEXT["PROJECT_ROOT"]
os.environ["GEMMA4_NOTEBOOK_DIR"] = str(NOTEBOOK_DIR)

# ---------- 推理格式配置 ----------
PROMPT_FORMAT = "normalized_xyxy"  # 推理格式: "normalized_xyxy"(归一化xyxy，推荐) 或 "legacy_1000x1000"(旧格式)
COORD_ORDER = "xyxy"  # 坐标顺序: "xyxy"(归一化格式) 或 "yxxy"(1000x1000旧格式)

# ---------- 模型路径配置 ----------
# 方式1: HuggingFace在线模型 (推荐, 自动下载)
# BASE_MODEL_PATH = "unsloth/gemma-4-E4B-it-bnb-4bit"
# 方式2: 本地已下载模型 (取消注释使用, 避免网络下载)
BASE_MODEL_PATH = "/raid5/sh/model/unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"

# ---------- 数据路径 (由01-数据预处理Notebook生成) ----------
# 自动定位训练数据文件: 优先查找OUTPUT_DIR下的jsonl, 其次查找scripts 目录
DATA_PATH = str(PROJECT_ROOT / "data" / "processed" / "unsloth_training_data-wgang_40" / "train.jsonl")
# 备选数据路径 (如果上述路径不存在, 使用此路径)
DATA_PATH_FALLBACK = str(PROJECT_ROOT / "data" / "sample" / "unsloth_train_data.jsonl")

# ---------- 模型输出路径 (层级化存储: 模型名/模式/时间戳) ----------
# 目录结构: models/finetuned/{model_name}/{mode}/{timestamp}/
# 例如: models/finetuned/gemma4_e4b_lora/ddp_8gpu/20260514_143528/
# 每次训练自动创建带时间戳的子目录, 不同模式的微调结果完全隔离
# 训练成功后自动写入latest.txt标记, 供03-推理和04-对比Notebook引用

from datetime import datetime

from unsloth_finetune.notebooking.train_shared import (
    MODE_DIR_MAP,
    build_lora_output_base,
    build_lora_output_dir,
    discover_eval_data_path,
    get_latest_output,
    resolve_eval_gpu_ids,
    resolve_latest_output_dir,
    resolve_mode_subdir,
)

TRAIN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

MODEL_NAME_SHORT = "gemma4_e4b_lora"
MODE_SUBDIR = resolve_mode_subdir(TRAINING_MODE)

LORA_OUTPUT_BASE = str(build_lora_output_base(PROJECT_ROOT, MODEL_NAME_SHORT))
LORA_OUTPUT_DIR = str(build_lora_output_dir(PROJECT_ROOT, TRAINING_MODE, TRAIN_TIMESTAMP, MODEL_NAME_SHORT))

LORA_OUTPUT_LATEST = get_latest_output(LORA_OUTPUT_BASE, TRAINING_MODE)

# ---------- 模型加载参数 ----------
MAX_SEQ_LENGTH = 2048  # 最大序列长度
LOAD_IN_4BIT = True  # 是否使用4-bit量化加载

# ---------- LoRA参数 ----------
# 视觉训练输入尺寸, 最终会透传给 dataset / train_distributed.py
IMAGE_WIDTH = 512
IMAGE_HEIGHT = 512
if IMAGE_WIDTH <= 0 or IMAGE_HEIGHT <= 0:
    raise ValueError(f"IMAGE_WIDTH 和 IMAGE_HEIGHT 必须为正整数: {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
TRAIN_IMAGE_SIZE = (IMAGE_WIDTH, IMAGE_HEIGHT)
LORA_R = 16  # LoRA秩, 控制可训练参数量 (推荐值: 16/32/64)
LORA_ALPHA = 16  # LoRA缩放因子 (通常设为等于LORA_R)
LORA_DROPOUT = 0  # Dropout率 (Unsloth推荐设为0)
RANDOM_STATE = 3407  # 随机种子

# ---------- 单GPU训练参数 (默认配置) ----------
PER_DEVICE_TRAIN_BATCH_SIZE = 8  # 单GPU每批次大小，给更高分辨率和长序列留出余量
GRADIENT_ACCUMULATION_STEPS = 2  # 梯度累积步数 (有效批次=batch_size*grad_accum)
WARMUP_RATIO = 0.1  # 预热比例 (取值范围: 0-1, 运行时自动转换为warmup_steps)
NUM_TRAIN_EPOCHS = 1  # 训练轮数
LEARNING_RATE = 2e-5  # 学习率
OPTIMIZER = "adamw_8bit"  # 优化器
WEIGHT_DECAY = 0.01  # 权重衰减
LR_SCHEDULER_TYPE = "linear"  # 学习率调度器类型
SEED = 3407  # 训练随机种子
LOGGING_STEPS = 10  # 日志记录步数
SAVE_STEPS = 300  # 模型保存步数
SAVE_TOTAL_LIMIT = 2  # 最多保存模型数量
TRAINING_OUTPUT_DIR = LORA_OUTPUT_DIR  # 统一把checkpoint与最终LoRA权重放到同一run目录
REPORT_TO = "none"  # 日志报告方式 ("none"/"wandb"/"tensorboard")

# ---------- 8x A6000 多GPU优化参数 ----------
MULTI_GPU_ENABLED = True
NUM_GPUS = 8
MULTI_GPU_BATCH_SIZE = 4
MULTI_GPU_GRAD_ACCUM = 2
MULTI_GPU_LR_BASE = 2e-5
MULTI_GPU_LR_SCALING = "none"
MULTI_GPU_WARMUP_RATIO = 0.1
MULTI_GPU_BF16 = True
MULTI_GPU_DISTRIBUTED_MODE = "ddp"
MULTI_GPU_GPU_MONITOR = True
MULTI_GPU_LOG_INTERVAL = 20

# ---------- CPU输入管线 / DataLoader优化 ----------
DATASET_MAX_WORKERS = 8
CPU_THREADS_PER_RANK = None
DATALOADER_NUM_WORKERS = None
DATALOADER_PREFETCH_FACTOR = None
DATALOADER_PIN_MEMORY = True
DATALOADER_PERSISTENT_WORKERS = True
DATALOADER_DROP_LAST = False
TF32_ENABLED = True
IMAGE_LOAD_MODE = "lazy"
IMAGE_BATCH_SIZE = None
MATERIALIZE_VISION_DATASET = False

# ---------- DDP 2倍吞吐参数 ----------
MODELS_PER_GPU = 1

# ---------- device_map 2D并行参数 ----------
GPU_GROUPS = [[0, 1], [2, 3], [4, 5], [6, 7]]
DEVICE_MAP_STRATEGY = "balanced"

# ---------- 自动检测模式参数 ----------
MODEL_VRAM_GB = 24.0

# ---------- GPU日志目录 ----------
SINGLE_GPU_LOG_DIR = str(PROJECT_ROOT / "notebooks" / "gpu_logs" / "single_gpu")
DDP_GPU_LOG_DIR = str(PROJECT_ROOT / "notebooks" / "gpu_logs" / "ddp_8gpu")
DDP_2X_GPU_LOG_DIR = str(PROJECT_ROOT / "notebooks" / "gpu_logs" / "ddp_8gpu_2x")
DEVICEMAP_GPU_LOG_DIR = str(PROJECT_ROOT / "notebooks" / "gpu_logs" / "devicemap_4group")
FSDP_GPU_LOG_DIR = str(PROJECT_ROOT / "notebooks" / "gpu_logs" / "fsdp_8gpu")
AUTO_GPU_LOG_DIR = str(PROJECT_ROOT / "notebooks" / "gpu_logs" / "auto_detect")
MULTI_NODE_GPU_LOG_DIR = str(PROJECT_ROOT / "notebooks" / "gpu_logs" / "multi_node")

GPU_LOG_DIR_MAP = {
    "single": SINGLE_GPU_LOG_DIR,
    "DDP": DDP_2X_GPU_LOG_DIR if MODELS_PER_GPU > 1 else DDP_GPU_LOG_DIR,
    "device_map": DEVICEMAP_GPU_LOG_DIR,
    "FSDP": FSDP_GPU_LOG_DIR,
    "auto": AUTO_GPU_LOG_DIR,
    "multi_node": MULTI_NODE_GPU_LOG_DIR,
}
CURRENT_GPU_LOG_DIR = GPU_LOG_DIR_MAP.get(TRAINING_MODE, SINGLE_GPU_LOG_DIR)

# ---------- 分布式训练脚本路径 ----------
TRAIN_SCRIPT_PATH = str(PROJECT_ROOT / "src" / "unsloth_finetune" / "training" / "distributed" / "train_distributed.py")

# ---------- 训练后统一评估 / 推理测试配置 ----------
ENABLE_PROMPT_SMOKE_TEST = True
POST_TRAIN_EVAL_ENABLED = True
POST_TRAIN_EVAL_AUTO_RUN = True
POST_TRAIN_EVAL_RAISE_ON_ERROR = True
POST_TRAIN_EVAL_IOU_THRESHOLD = 0.5
POST_TRAIN_EVAL_BATCH_SIZE = 4
POST_TRAIN_EVAL_MAX_EVAL_SAMPLES = None
POST_TRAIN_EVAL_MAX_NEW_TOKENS = 512
POST_TRAIN_EVAL_TEMPERATURE = 0.7
POST_TRAIN_EVAL_TOP_P = 0.9
POST_TRAIN_EVAL_SCHEDULER_MODE = "dynamic_queue"
POST_TRAIN_EVAL_PARTITION_STRATEGY = "round_robin"
POST_TRAIN_EVAL_EXPORT_LABELME = False
POST_TRAIN_EVAL_TORCHRUN_PORT = 29510
POST_TRAIN_EVAL_WORKDIR = str(PROJECT_ROOT)
POST_TRAIN_EVAL_SCRIPT_PATH = str(PROJECT_ROOT / "src" / "unsloth_finetune" / "training" / "distributed" / "distributed_inference.py")
POST_TRAIN_EVAL_PROGRESS_RANKS = "0"
POST_TRAIN_EVAL_RESULT_DIR = str(Path(LORA_OUTPUT_DIR) / "post_training_eval")
POST_TRAIN_EVAL_SUMMARY = {}
POST_TRAIN_EVAL_FILES = {}
POST_TRAIN_EVAL_LAST_RETURN_CODE = None
MULTI_GPU_TRAINING_MODES = {"DDP", "device_map", "FSDP", "auto", "multi_node"}


POST_TRAIN_EVAL_DATA_PATH, POST_TRAIN_EVAL_SPLIT_NAME = discover_eval_data_path(DATA_PATH)
POST_TRAIN_EVAL_GPU_IDS = resolve_eval_gpu_ids(TRAINING_MODE, torch.cuda.device_count(), GPU_GROUPS)
POST_TRAIN_EVAL_MODE = (
    "multi_gpu"
    if TRAINING_MODE in MULTI_GPU_TRAINING_MODES and len(POST_TRAIN_EVAL_GPU_IDS) > 1
    else "single_gpu"
)

if MULTI_GPU_ENABLED and torch.cuda.device_count() >= 2:
    _world_size = torch.cuda.device_count()
    _ddp_effective_batch = MULTI_GPU_BATCH_SIZE * MULTI_GPU_GRAD_ACCUM * _world_size
    _notebook_effective_batch = MULTI_GPU_BATCH_SIZE * MULTI_GPU_GRAD_ACCUM
    if MULTI_GPU_LR_SCALING == "linear":
        _ddp_effective_lr = MULTI_GPU_LR_BASE * _world_size
    elif MULTI_GPU_LR_SCALING == "sqrt":
        _ddp_effective_lr = MULTI_GPU_LR_BASE * (_world_size**0.5)
    else:
        _ddp_effective_lr = MULTI_GPU_LR_BASE
    print(f"\n{'=' * 50}")
    print("多GPU优化配置")
    print(f"{'=' * 50}")
    print(f"GPU数量: {_world_size}")
    print(f"每GPU批次: {MULTI_GPU_BATCH_SIZE}")
    print(f"梯度累积: {MULTI_GPU_GRAD_ACCUM}")
    print(f"DDP有效批次: {_ddp_effective_batch}")
    print(f"Notebook有效批次(单进程): {_notebook_effective_batch}")
    print(f"DDP学习率: {_ddp_effective_lr:.6f} ({MULTI_GPU_LR_SCALING}缩放)")
    print(f"Notebook学习率: {MULTI_GPU_LR_BASE:.6f} (不缩放)")
    print(f"混合精度: BF16={MULTI_GPU_BF16}")
    print(f"分布式模式: {MULTI_GPU_DISTRIBUTED_MODE}")
    print(f"GPU监控: {MULTI_GPU_GPU_MONITOR}")
else:
    print(f"\n单GPU模式 (检测到 {torch.cuda.device_count()} GPU)")
    print(f"有效批次: {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
    print(f"学习率: {LEARNING_RATE}")

import os

if os.path.exists(DATA_PATH):
    ACTUAL_DATA_PATH = DATA_PATH
    print(f"使用训练数据: {DATA_PATH}")
elif os.path.exists(DATA_PATH_FALLBACK):
    ACTUAL_DATA_PATH = DATA_PATH_FALLBACK
    print(f"使用备选数据: {DATA_PATH_FALLBACK}")
else:
    ACTUAL_DATA_PATH = DATA_PATH_FALLBACK
    print(f"警告: 数据文件不存在，将使用备选路径: {DATA_PATH_FALLBACK}")

POST_TRAIN_EVAL_DATA_PATH, POST_TRAIN_EVAL_SPLIT_NAME = discover_eval_data_path(ACTUAL_DATA_PATH)
POST_TRAIN_EVAL_RESULT_DIR = str(Path(resolve_latest_output_dir(LORA_OUTPUT_BASE, TRAINING_MODE, LORA_OUTPUT_DIR)) / "post_training_eval")

print(f"项目根目录: {PROJECT_ROOT}")
print(f"基础模型: {BASE_MODEL_PATH}")
print(f"LoRA输出(当前): {LORA_OUTPUT_DIR}")
print(f"LoRA输出(基础): {LORA_OUTPUT_BASE}")
print(f"训练时间戳: {TRAIN_TIMESTAMP}")
if LORA_OUTPUT_LATEST:
    print(f"LoRA输出(最新): {LORA_OUTPUT_LATEST}")
else:
    print("LoRA输出(最新): 无历史训练记录")
print(f"训练中间状态输出目录: {TRAINING_OUTPUT_DIR}")
print(f"GPU监控目录: {CURRENT_GPU_LOG_DIR}")
if POST_TRAIN_EVAL_DATA_PATH:
    print(f"标准化评估数据: {POST_TRAIN_EVAL_DATA_PATH} (split={POST_TRAIN_EVAL_SPLIT_NAME})")
else:
    print("标准化评估数据: 未发现 test/valid/train JSONL, 将仅执行 smoke test")
print(f"标准化评估模式: {POST_TRAIN_EVAL_MODE}")
print(f"标准化评估GPU: {POST_TRAIN_EVAL_GPU_IDS}")
print(f"标准化评估结果目录: {POST_TRAIN_EVAL_RESULT_DIR}")

训练模式: DDP

多GPU优化配置
GPU数量: 8
每GPU批次: 8
梯度累积: 2
DDP有效批次(8卡): 128
Notebook有效批次(单进程): 16
DDP学习率: 0.000320 (linear缩放)
Notebook学习率: 0.000040 (不缩放)
混合精度: BF16=True
分布式模式: ddp
GPU监控: True
使用训练数据: /raid5/sh/code/unsloth-finetune/data/processed/unsloth_training_data-wgang_40/train.jsonl
项目根目录: /raid5/sh/code/unsloth-finetune
基础模型: /raid5/sh/model/unsloth/gemma-4-E4B-it-unsloth-bnb-4bit
LoRA输出(当前): /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora/ddp_8gpu/20260521_073055
LoRA输出(基础): /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora
训练时间戳: 20260521_073055
LoRA输出(最新): 无历史训练记录
训练中间状态输出目录: /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora/ddp_8gpu/20260521_073055
GPU监控目录: /raid5/sh/code/unsloth-finetune/notebooks/gpu_logs/ddp_8gpu
标准化评估数据: /raid5/sh/code/unsloth-finetune/data/processed/unsloth_training_data-wgang_40/test.jsonl (split=test)
标准化评估模式: multi_gpu
标准化评估GPU: [0, 1, 2, 3, 4, 5, 6, 7]
标准化评估结果目录: /raid5/sh/code/unsloth-finetune/models/finetuned/ge

In [ ]:
from pathlib import Path

import json as _json


def _print_file_info(label: str, path_value: str | None):
    if not path_value:
        print(f"{label}: None")
        return None

    path = Path(path_value)
    exists = path.exists()
    if exists:
        size_mb = path.stat().st_size / 1024 / 1024
        print(f"{label}: {path} (exists=True, size={size_mb:.2f}MB)")
    else:
        print(f"{label}: {path} (exists=False)")
    return path


def _sample_jsonl(path: Path, n: int = 5) -> tuple[int, int]:
    ok = 0
    bad = 0
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                _json.loads(line)
                ok += 1
            except Exception:
                bad += 1
            if ok + bad >= n:
                break
    return ok, bad


print("\n" + "=" * 60)
print("训练前检查")
print("=" * 60)

import torch

train_data_path = ACTUAL_DATA_PATH if "ACTUAL_DATA_PATH" in globals() else DATA_PATH
train_file = _print_file_info("训练数据", train_data_path)
if train_file and train_file.exists():
    ok, bad = _sample_jsonl(train_file)
    print(f"训练数据JSONL抽样: ok={ok}, bad={bad}")
    if ok <= 0:
        raise ValueError(f"训练数据文件没有可解析的JSONL行: {train_file}")
else:
    raise FileNotFoundError(f"训练数据文件不存在: {train_data_path}")

eval_file = _print_file_info("评估数据", POST_TRAIN_EVAL_DATA_PATH)
print(f"评估split: {POST_TRAIN_EVAL_SPLIT_NAME}")
if eval_file and eval_file.exists() and train_file and train_file.exists():
    if eval_file.resolve() == train_file.resolve():
        print("警告: 评估数据与训练数据相同，可能导致指标偏高或口径不一致")

gpu_count = int(torch.cuda.device_count())
print(f"检测到GPU: {gpu_count}")

use_multi = TRAINING_MODE in MULTI_GPU_TRAINING_MODES and gpu_count > 1
if use_multi:
    per_device_bs = int(MULTI_GPU_BATCH_SIZE)
    grad_accum = int(MULTI_GPU_GRAD_ACCUM)
    base_lr = float(MULTI_GPU_LR_BASE)
    lr_scaling = str(MULTI_GPU_LR_SCALING)
    world_size = int(len(GPU_GROUPS)) if TRAINING_MODE == "device_map" else gpu_count
else:
    per_device_bs = int(PER_DEVICE_TRAIN_BATCH_SIZE)
    grad_accum = int(GRADIENT_ACCUMULATION_STEPS)
    base_lr = float(LEARNING_RATE)
    lr_scaling = "none"
    world_size = 1

effective_batch = per_device_bs * grad_accum * world_size
if lr_scaling == "linear":
    effective_lr = base_lr * world_size
elif lr_scaling == "sqrt":
    effective_lr = base_lr * (world_size**0.5)
else:
    effective_lr = base_lr

print(f"per_device_batch_size: {per_device_bs}")
print(f"gradient_accumulation_steps: {grad_accum}")
print(f"world_size: {world_size}")
print(f"effective_global_batch: {effective_batch}")
print(f"learning_rate_base: {base_lr}")
print(f"lr_scaling: {lr_scaling}")
print(f"learning_rate_effective: {effective_lr}")

if use_multi and effective_lr > 5e-4:
    raise ValueError(
        "effective_lr过高，疑似发生学习率重复缩放。"
        f"当前 base_lr={base_lr}, effective_lr={effective_lr}, world_size={world_size}, lr_scaling={lr_scaling}。"
        "请确保Notebook侧传入未缩放的 base learning rate。"
    )

latest_file = Path(LORA_OUTPUT_BASE) / MODE_SUBDIR / "latest.txt"
print(f"latest标记文件: {latest_file} (exists={latest_file.exists()})")
if latest_file.exists():
    latest_ts = latest_file.read_text(encoding="utf-8").strip()
    latest_dir = Path(LORA_OUTPUT_BASE) / MODE_SUBDIR / latest_ts if latest_ts else None
    print(f"latest指向: {latest_dir} (exists={latest_dir.exists() if latest_dir else False})")

output_dir = Path(LORA_OUTPUT_DIR)
print(f"本次输出目录: {output_dir}")
if output_dir.exists():
    existing_items = list(output_dir.iterdir())
    print(f"警告: 输出目录已存在，子项数={len(existing_items)}")


## 2. 加载模型

使用 Unsloth 的 `FastVisionModel` 加载 Gemma 4-E4B 视觉语言模型，采用 4-bit QLoRA 量化。

### device_map 策略

⚠️ **关键：`device_map`决定模型是分片还是完整加载**

- `device_map="balanced"/"auto"` → **模型并行**: 模型拆分到多卡, 仅1个进程, **无法实现多GPU加速**
- `device_map={"": 0}` → **单卡完整加载**: 整个模型在GPU 0, Notebook模式推荐
- 不设置 `device_map` → DDP模式下每进程独立加载到local_rank GPU, **数据并行8x加速**

本Notebook自动检测: Notebook用`{"": 0}`, DDP环境不设置

### 模型路径配置

- `BASE_MODEL_PATH` 已在上方配置单元格中定义
- 默认使用 HuggingFace 在线路径，自动下载模型
- 如需使用本地模型，请在配置单元格中修改为本地路径


In [5]:
# 模型配置参数
# ============================================================
# 所有参数已在上方配置单元格中集中定义, 此处仅引用

if TRAINING_MODE != "single":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过模型加载 (分布式训练由train_distributed.py处理)")
else:
    os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

    if os.path.exists(BASE_MODEL_PATH):
        print(f"使用本地模型: {BASE_MODEL_PATH}")
    else:
        print(f"将从HuggingFace下载模型: {BASE_MODEL_PATH}")

    is_distributed = os.environ.get("LOCAL_RANK") is not None
    _device_map = None if is_distributed else {"": 0}

    print(f"正在加载模型: {BASE_MODEL_PATH}")
    print(f"device_map={_device_map} (Notebook=单卡完整加载, DDP=每进程独立GPU)")

    model, processor = FastVisionModel.from_pretrained(
        model_name=BASE_MODEL_PATH,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        dtype=None,
        device_map=_device_map,
        disable_log_stats=True,
    )

    tokenizer = processor.tokenizer

    print("视觉模型加载完成！")
    print(f"模型参数量: {model.num_parameters() / 1e9:.2f}B")
    print(f"处理器类型: {type(processor).__name__}")
    print(f"Tokenizer类型: {type(tokenizer).__name__}")

⏭ 当前模式: DDP, 跳过模型加载 (分布式训练由train_distributed.py处理)


## 3. 配置 LoRA 适配器

添加 LoRA adapters，只更新少量参数，大幅降低训练成本。

### LoRA 参数说明

| 参数           | 说明                     | 推荐值             |
| -------------- | ------------------------ | ------------------ |
| r              | LoRA秩，控制可训练参数量 | 16 (默认) 或 32-64 |
| lora_alpha     | 缩放因子                 | 通常设为 r         |
| lora_dropout   | Dropout率                | Unsloth推荐0       |
| target_modules | 目标模块                 | attention + FFN    |


In [6]:
if TRAINING_MODE != "single":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过LoRA配置")
else:
    model = FastVisionModel.get_peft_model(
        model,
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_percent = trainable_params / total_params * 100

    print(f"可训练参数: {trainable_params:,} ({trainable_percent:.2f}%)")
    print(f"总参数: {total_params:,}")

    # ============================================================
    # 梯度检查点与缓存兼容性处理
    # ============================================================
    # Gemma4 模型的 KV 缓存与梯度检查点不兼容
    # 训练时需要禁用缓存以避免警告和潜在问题
    gc_enabled = getattr(model, "gradient_checkpointing", False)
    if hasattr(model, "gradient_checkpointing_enable"):
        gc_enabled = True

    cache_status = getattr(model.config, "use_cache", None)
    if gc_enabled and cache_status:
        model.config.use_cache = False
        print(f"\n✅ 梯度检查点兼容性处理:")
        print(f"   检测到梯度检查点已启用")
        print(f"   缓存状态: {cache_status} → False (已禁用)")
        print(f"   原因: KV缓存与梯度检查点不兼容，训练时需禁用")
    elif gc_enabled:
        print(f"\n✅ 梯度检查点兼容性检查:")
        print(f"   检测到梯度检查点已启用")
        print(f"   缓存状态: {cache_status} (已为正确配置)")
    else:
        print(f"\n⚠️ 梯度检查点未启用")
        print(f"   如需启用梯度检查点，请在 LoRA 配置中设置 use_gradient_checkpointing='unsloth'")

⏭ 当前模式: DDP, 跳过LoRA配置


## 4. 准备训练数据

支持多种数据格式：

1. **Messages格式** - Gemma 4原生对话格式
2. **Alpaca格式** - 通用指令格式
3. **自定义JSONL** - 自定义格式

### 数据格式示例

```json
{
  "messages": [
    { "role": "user", "content": "用户问题" },
    { "role": "assistant", "content": "回答" }
  ]
}
```


In [7]:
if TRAINING_MODE != "single":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过数据加载")
else:
    import sys

    sys.path.insert(0, str(PROJECT_ROOT))

    from unsloth_finetune.training.distributed.dataset import MultimodalDataset, print_memory_status

    DATA_PATH = ACTUAL_DATA_PATH

    data_file = Path(DATA_PATH)
    if data_file.exists():
        print(f"数据文件存在: {DATA_PATH}")
        print_memory_status("加载前内存")
        mm_dataset = MultimodalDataset(
            data_path=DATA_PATH,
            image_load_mode=IMAGE_LOAD_MODE,
            max_workers=DATASET_MAX_WORKERS,
            show_progress=True,
            batch_size=IMAGE_BATCH_SIZE,
            image_size=TRAIN_IMAGE_SIZE,
        )
        stats = mm_dataset.stats()
        print(f"\n数据集统计信息:")
        print(f"  数据集大小: {stats['total_samples']} 条")
        print(f"  含图片样本: {stats['samples_with_images']} 条")
        print(f"  图片总数: {stats['total_image_paths']} 张")
        print(f"  唯一图片路径: {stats['unique_image_paths']} 个")
        print(f"  图片加载模式: {stats['image_load_mode']}")
        print(f"  进程内存: {stats['memory_rss_gb']} GB")
        print(f"  训练图片尺寸: {stats['configured_image_size']}")
        print(f"  数据集供给方式: {'预物化list' if MATERIALIZE_VISION_DATASET else '懒加载Dataset'}")
    else:
        print(f"数据文件不存在: {DATA_PATH}")
        raise FileNotFoundError(f"训练数据文件不存在: {DATA_PATH}")

    print("\n数据样本预览:")
    sample = mm_dataset[0]
    print("样本 messages 结构:")
    for msg in sample["messages"]:
        content_types = [item.get("type") for item in msg["content"]]
        print(f"  {msg['role']}: content types = {content_types}")
    user_content = sample["messages"][0]["content"]
    for item in user_content:
        if item.get("type") == "image" and "image" in item:
            img = item["image"]
            print(f"图片对象: {type(img).__name__}, size={img.size}, expected={TRAIN_IMAGE_SIZE}")
            break


⏭ 当前模式: DDP, 跳过数据加载


In [8]:
# 数据预处理
# ============================================================
if TRAINING_MODE != "single":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过数据转换")
else:
    if MATERIALIZE_VISION_DATASET:
        train_dataset = mm_dataset.to_conversation_list()
    else:
        train_dataset = mm_dataset

    print("\n数据预处理完成！")
    print(f"数据集类型: {type(train_dataset).__name__}")
    print(f"数据集大小: {len(train_dataset)} 条")
    print(f"供给策略: {'预物化list' if MATERIALIZE_VISION_DATASET else '按batch懒加载'}")
    print_memory_status("转换后内存")

    sample = train_dataset[0]
    print("\n样本结构验证:")
    print(f"  keys: {list(sample.keys())}")
    print(f"  期望训练图片尺寸: {TRAIN_IMAGE_SIZE}")
    print(f"  messages 数量: {len(sample['messages'])}")
    for msg in sample["messages"]:
        print(f"  {msg['role']}: content items = {len(msg['content'])}")
        for item in msg["content"]:
            if item.get("type") == "image":
                img = item.get("image")
                actual_size = img.size if img else "N/A"
                print(f"    - image: {type(img).__name__}, size={actual_size}")
                if img is not None and actual_size != TRAIN_IMAGE_SIZE:
                    raise ValueError(
                        f"训练图片尺寸校验失败: 期望 {TRAIN_IMAGE_SIZE}, 实际 {actual_size}"
                    )
            elif item.get("type") == "text":
                print(f"    - text: {item.get('text', '')[:50]}...")

    print("\n提示: 训练时需要使用 UnslothVisionDataCollator")


⏭ 当前模式: DDP, 跳过数据转换


## 5. 配置训练参数

使用 `SFTTrainer` 进行监督微调训练。


In [9]:
if TRAINING_MODE != "single":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过训练器配置")
else:
    import sys

    from unsloth.trainer import UnslothVisionDataCollator

    sys.path.insert(0, str(PROJECT_ROOT))
    from unsloth_finetune.training.distributed.gpu_monitor import GPUMonitor, GPUMonitorCallback

    n_gpus = torch.cuda.device_count()
    is_ddp_process = os.environ.get("LOCAL_RANK") is not None

    if TF32_ENABLED and torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
        torch.set_float32_matmul_precision("high")

    if MULTI_GPU_ENABLED and n_gpus >= 2:
        if is_ddp_process:
            _batch_size = MULTI_GPU_BATCH_SIZE
            _grad_accum = MULTI_GPU_GRAD_ACCUM
            if MULTI_GPU_LR_SCALING == "linear":
                _lr = MULTI_GPU_LR_BASE * n_gpus
            elif MULTI_GPU_LR_SCALING == "sqrt":
                _lr = MULTI_GPU_LR_BASE * (n_gpus**0.5)
            else:
                _lr = MULTI_GPU_LR_BASE
            _warmup_ratio = MULTI_GPU_WARMUP_RATIO
            _bf16 = MULTI_GPU_BF16 and torch.cuda.is_bf16_supported()
            _effective_batch = _batch_size * _grad_accum * n_gpus
            print(f"DDP多GPU配置 ({n_gpus} GPU, 每卡完整模型):")
            print(f"  batch_size={_batch_size}, grad_accum={_grad_accum}")
            print(f"  lr={_lr:.6f} ({MULTI_GPU_LR_SCALING}缩放)")
            print(f"  BF16={_bf16}")
            print(f"  有效批次={_effective_batch}")
        else:
            _batch_size = PER_DEVICE_TRAIN_BATCH_SIZE
            _grad_accum = GRADIENT_ACCUMULATION_STEPS
            _lr = LEARNING_RATE
            _warmup_ratio = WARMUP_RATIO
            _bf16 = torch.cuda.is_bf16_supported()
            _effective_batch = _batch_size * _grad_accum
            print(f"⚠️ Notebook单进程模式 (检测到{n_gpus}GPU, 但DDP需torchrun)")
            _device_map_desc = "{'': 0}" if not is_ddp_process else "None (每进程独立GPU)"
            print(f"  模型仅使用GPU 0 (device_map={_device_map_desc})")
            print(f"  batch_size={_batch_size}, grad_accum={_grad_accum}")
            print(f"  lr={_lr:.6f} (不缩放, DDP才需LR缩放)")
            print(f"  BF16={_bf16}")
            print(f"  有效批次={_effective_batch} (DDP={_effective_batch * n_gpus})")
            print("  建议真正训练时通过 torchrun 调用 scripts/train_distributed.py")
    else:
        _batch_size = PER_DEVICE_TRAIN_BATCH_SIZE
        _grad_accum = GRADIENT_ACCUMULATION_STEPS
        _lr = LEARNING_RATE
        _warmup_ratio = WARMUP_RATIO
        _bf16 = torch.cuda.is_bf16_supported()
        _effective_batch = _batch_size * _grad_accum
        print(f"单GPU配置 ({n_gpus} GPU):")
        print(f"  batch_size={_batch_size}, grad_accum={_grad_accum}")
        print(f"  lr={_lr}, BF16={_bf16}")
        print(f"  有效批次={_effective_batch}")

    _warmup_steps = max(1, int(len(train_dataset) * NUM_TRAIN_EPOCHS / _effective_batch * _warmup_ratio))
    print(f"  warmup: {_warmup_ratio} ratio -> {_warmup_steps} steps")

    TRAINING_CONFIG = {
        "per_device_train_batch_size": _batch_size,
        "gradient_accumulation_steps": _grad_accum,
        "warmup_steps": _warmup_steps,
        "num_train_epochs": NUM_TRAIN_EPOCHS,
        "learning_rate": _lr,
        "optim": OPTIMIZER,
        "weight_decay": WEIGHT_DECAY,
        "lr_scheduler_type": LR_SCHEDULER_TYPE,
        "seed": SEED,
        "logging_steps": LOGGING_STEPS,
        "save_strategy": "steps",
        "save_steps": SAVE_STEPS,
        "save_total_limit": SAVE_TOTAL_LIMIT,
        "output_dir": TRAINING_OUTPUT_DIR,
        "report_to": REPORT_TO,
        "bf16": _bf16,
        "fp16": not _bf16,
        "remove_unused_columns": False,
        "dataset_text_field": "",
        "dataloader_pin_memory": DATALOADER_PIN_MEMORY,
        "dataloader_drop_last": DATALOADER_DROP_LAST,
        "save_safetensors": True,
    }

    if DATALOADER_NUM_WORKERS is not None:
        TRAINING_CONFIG["dataloader_num_workers"] = DATALOADER_NUM_WORKERS
    if DATALOADER_NUM_WORKERS and DATALOADER_NUM_WORKERS > 0:
        TRAINING_CONFIG["dataloader_persistent_workers"] = DATALOADER_PERSISTENT_WORKERS
        if DATALOADER_PREFETCH_FACTOR is not None:
            TRAINING_CONFIG["dataloader_prefetch_factor"] = DATALOADER_PREFETCH_FACTOR

    gpu_monitor = GPUMonitor(
        log_dir=CURRENT_GPU_LOG_DIR,
        log_interval=LOGGING_STEPS,
    )
    gpu_callback = GPUMonitorCallback(gpu_monitor, print_interval=LOGGING_STEPS)

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        processing_class=processor.tokenizer,
        data_collator=UnslothVisionDataCollator(model, processor),
        args=SFTConfig(
            **TRAINING_CONFIG,
            max_seq_length=MAX_SEQ_LENGTH,
        ),
        callbacks=[gpu_callback],
    )

    print("\n视觉模型训练器创建完成！")
    print("使用 UnslothVisionDataCollator 进行图片处理")
    print(f"数据集类型: {type(train_dataset).__name__}")
    print(f"数据集大小: {len(train_dataset)} 条")
    print(f"有效批次大小: {_effective_batch}")
    print(f"训练图片尺寸: {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
    print(f"TF32: {TF32_ENABLED}")
    print(f"DataLoader workers: {DATALOADER_NUM_WORKERS}")
    print(f"GPU监控: 已集成 (日志目录: {CURRENT_GPU_LOG_DIR}/)")
    print(f"统一训练产物目录: {TRAINING_OUTPUT_DIR}")


⏭ 当前模式: DDP, 跳过训练器配置


## 6. 开始训练

执行训练过程，监控loss变化。


In [10]:
if TRAINING_MODE != "single":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过GPU内存状态检查")
else:
    # 显示GPU内存状态
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU: {gpu_stats.name}")
    print(f"初始VRAM使用: {start_gpu_memory} GB / {max_memory} GB")
    print(f"剩余VRAM: {max_memory - start_gpu_memory} GB")

⏭ 当前模式: DDP, 跳过GPU内存状态检查


In [11]:
if TRAINING_MODE != "single":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过训练执行")
else:
    # 开始训练
    print("开始训练...")
    trainer_stats = trainer.train()

    print("\n训练完成！")

    # 获取训练统计信息（使用.get()避免KeyError）
    metrics = trainer_stats.metrics

    train_runtime = metrics.get("train_runtime", 0)
    train_loss = metrics.get("train_loss", 0)
    train_steps = metrics.get("train_steps", 0)

    # 计算实际样本数
    effective_batch_size = TRAINING_CONFIG["per_device_train_batch_size"] * TRAINING_CONFIG["gradient_accumulation_steps"]
    total_samples = len(train_dataset) if hasattr(train_dataset, "__len__") else 0

    print(f"训练时长: {train_runtime:.2f} 秒")
    print(f"训练步数: {train_steps}")
    print(f"最终loss: {train_loss:.4f}")
    print(f"数据集样本数: {total_samples}")
    print(f"有效批次大小: {effective_batch_size}")

    # 显示所有可用的metrics
    print("\n所有训练统计信息:")
    for key, value in metrics.items():
        print(f"  {key}: {value}")

⏭ 当前模式: DDP, 跳过训练执行


In [12]:
if TRAINING_MODE != "single":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过训练后GPU内存检查")
else:
    # 显示训练后GPU内存状态
    used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
    used_percentage = round(used_memory / max_memory * 100, 3)
    lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

    print(f"训练后VRAM使用: {used_memory} GB")
    print(f"LoRA占用VRAM: {used_memory_for_lora} GB")
    print(f"VRAM使用率: {used_percentage}%")
    print(f"LoRA VRAM占用率: {lora_percentage}%")

⏭ 当前模式: DDP, 跳过训练后GPU内存检查


## 7. 推理测试

兼容两类验证路径：

- 单GPU：保留原有内存态 prompt smoke test
- 单GPU/多GPU：统一支持基于 `distributed_inference.py` 的标准化评估、结果聚合与指标统计

In [13]:
from statistics import pstdev
from typing import Any, Dict, List


def infer_generation_device(model_obj) -> torch.device:
    try:
        return next(model_obj.parameters()).device
    except StopIteration:
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def generate_chat_response(model_obj, tokenizer_obj, prompt, max_new_tokens=128, **generate_kwargs):
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    device = infer_generation_device(model_obj)
    inputs = tokenizer_obj.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(device)
    outputs = model_obj.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        **generate_kwargs,
    )
    response = tokenizer_obj.batch_decode(outputs, skip_special_tokens=True)
    return response[0]


def aggregate_metric_dicts(sample_metrics_list: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not sample_metrics_list:
        return {}
    keys = ["precision", "recall", "f1", "num_det", "num_gt", "num_match", "mean_match_iou"]
    result = {}
    for key in keys:
        values = [float(m[key]) for m in sample_metrics_list if key in m]
        if not values:
            result[f"mean_{key}"] = 0.0
            result[f"std_{key}"] = 0.0
            continue
        mean_value = sum(values) / len(values)
        result[f"mean_{key}"] = mean_value
        result[f"std_{key}"] = pstdev(values) if len(values) > 1 else 0.0
    success_flags = [1.0 if m.get("det_success", False) else 0.0 for m in sample_metrics_list]
    result["total_samples"] = len(sample_metrics_list)
    result["success_rate"] = sum(success_flags) / len(success_flags) if success_flags else 0.0
    return result


class PostTrainingEvaluationConfig:
    VALID_MODES = ("single_gpu", "multi_gpu")

    def __init__(
        self,
        mode: str,
        gpu_ids: List[int],
        base_model_path: str,
        lora_adapter_path: str,
        data_path: str,
        result_dir: str,
        worker_script_path: str,
        workdir: str,
        max_seq_length: int,
        load_in_4bit: bool,
        iou_threshold: float,
        batch_size: int,
        max_eval_samples: int,
        max_new_tokens: int,
        temperature: float,
        top_p: float,
        scheduler_mode: str,
        partition_strategy: str,
        export_labelme: bool,
        torchrun_port: int,
        prompt_format: str = "normalized_xyxy",
        coord_order: str = "xyxy",
    ):
        if mode not in self.VALID_MODES:
            raise ValueError(f"无效评估模式: {mode}, 有效值: {self.VALID_MODES}")
        if not gpu_ids:
            raise ValueError("评估至少需要1张GPU")
        self.mode = mode
        self.gpu_ids = [int(g) for g in gpu_ids]
        self.base_model_path = str(base_model_path)
        self.lora_adapter_path = str(lora_adapter_path)
        self.data_path = str(data_path)
        self.result_dir = str(result_dir)
        self.worker_script_path = str(worker_script_path)
        self.workdir = str(workdir)
        self.max_seq_length = int(max_seq_length)
        self.load_in_4bit = bool(load_in_4bit)
        self.iou_threshold = float(iou_threshold)
        self.batch_size = int(batch_size)
        self.max_eval_samples = None if max_eval_samples is None else int(max_eval_samples)
        self.max_new_tokens = int(max_new_tokens)
        self.temperature = float(temperature)
        self.top_p = float(top_p)
        self.scheduler_mode = str(scheduler_mode)
        self.partition_strategy = str(partition_strategy)
        self.export_labelme = bool(export_labelme)
        self.torchrun_port = int(torchrun_port)
        self.prompt_format = str(prompt_format)
        self.coord_order = str(coord_order)

    @property
    def visible_gpu_csv(self) -> str:
        return ",".join(str(g) for g in self.gpu_ids)

    def ensure_worker_script(self) -> Path:
        script_path = Path(self.worker_script_path)
        if not script_path.exists():
            raise FileNotFoundError(f"评估脚本不存在: {script_path}")
        return script_path

    def build_command_parts(self) -> List[str]:
        script_path = self.ensure_worker_script()
        common_parts = [
            "--gpu_ids",
            self.visible_gpu_csv,
            "--base_model_path",
            self.base_model_path,
            "--lora_adapter_path",
            self.lora_adapter_path,
            "--data_path",
            self.data_path,
            "--result_dir",
            self.result_dir,
            "--max_seq_length",
            str(self.max_seq_length),
            "--batch_size",
            str(self.batch_size),
            "--iou_threshold",
            str(self.iou_threshold),
            "--max_new_tokens",
            str(self.max_new_tokens),
            "--temperature",
            str(self.temperature),
            "--top_p",
            str(self.top_p),
            "--scheduler_mode",
            self.scheduler_mode,
            "--partition_strategy",
            self.partition_strategy,
            "--prompt_format",
            self.prompt_format,
            "--coord_order",
            self.coord_order,
        ]
        if self.load_in_4bit:
            common_parts.append("--load_in_4bit")
        if self.export_labelme:
            common_parts.append("--export_labelme")
        if self.max_eval_samples is not None:
            common_parts.extend(["--max_eval_samples", str(self.max_eval_samples)])
        if self.mode == "single_gpu":
            return [sys.executable, str(script_path), *common_parts]
        return [
            "torchrun",
            f"--nproc_per_node={len(self.gpu_ids)}",
            "--rdzv_backend=c10d",
            f"--rdzv_endpoint=localhost:{self.torchrun_port}",
            str(script_path),
            *common_parts,
        ]

    def format_command_preview(self, command_parts: List[str]) -> str:
        preview_parts = [shlex.quote(str(part)) for part in command_parts]
        lines = []
        for index, part in enumerate(preview_parts):
            suffix = " \\" if index < len(preview_parts) - 1 else ""
            if index == 0:
                lines.append(f"{part}{suffix}")
            else:
                lines.append(f"  {part}{suffix}")
        return "\n".join(lines)

    def print_summary(self):
        print("=" * 72)
        print("训练后标准化评估摘要")
        print("=" * 72)
        print(f"评估模式: {self.mode}")
        print(f"GPU IDs: {self.gpu_ids}")
        print(f"数据集: {self.data_path}")
        print(f"LoRA目录: {self.lora_adapter_path}")
        print(f"结果目录: {self.result_dir}")
        print(f"worker脚本: {self.worker_script_path}")
        print(f"工作目录: {self.workdir}")
        print(f"batch_size: {self.batch_size}")
        print(f"scheduler_mode: {self.scheduler_mode}")
        print(f"partition_strategy: {self.partition_strategy}")
        print(f"max_eval_samples: {self.max_eval_samples}")
        print(f"prompt_format: {self.prompt_format}")
        print(f"coord_order: {self.coord_order}")
        print(f"自动执行: {POST_TRAIN_EVAL_AUTO_RUN}")
        print("=" * 72)


def build_post_training_eval_env(config: PostTrainingEvaluationConfig) -> Dict[str, str]:
    env = os.environ.copy()
    pythonpath_entries = [str(PROJECT_ROOT)]
    if env.get("PYTHONPATH"):
        pythonpath_entries.append(env["PYTHONPATH"])
    env["PYTHONPATH"] = os.pathsep.join(pythonpath_entries)
    env["CUDA_VISIBLE_DEVICES"] = config.visible_gpu_csv
    env.setdefault("GEMMA4_NOTEBOOK_DIR", str(NOTEBOOK_DIR))
    env.setdefault("GEMMA4_LIVE_TQDM_RANKS", POST_TRAIN_EVAL_PROGRESS_RANKS)
    notebook_file = NOTEBOOK_CONTEXT.get("NOTEBOOK_FILE", "")
    if notebook_file:
        env["GEMMA4_NOTEBOOK_FILE"] = notebook_file
    return env


def run_post_training_evaluation(config: PostTrainingEvaluationConfig, raise_on_error: bool = True) -> int:
    command_parts = config.build_command_parts()
    env = build_post_training_eval_env(config)
    result_dir = Path(config.result_dir)
    result_dir.mkdir(parents=True, exist_ok=True)
    print("\n开始执行训练后标准化评估...")
    print(f"CUDA_VISIBLE_DEVICES={env.get('CUDA_VISIBLE_DEVICES', '')}")
    print("命令预览:")
    print(config.format_command_preview(command_parts))
    completed = subprocess.run(
        command_parts,
        cwd=config.workdir,
        env=env,
        check=False,
    )
    return_code = int(completed.returncode)
    print(f"评估进程已结束, return code = {return_code}")
    if return_code != 0 and raise_on_error:
        raise RuntimeError(f"训练后标准化评估失败, return code = {return_code}")
    return return_code


def load_post_training_evaluation_outputs(result_dir: str) -> Dict[str, Any]:
    result_dir_path = Path(result_dir)
    ft_path = result_dir_path / "finetuned_results.json"
    base_path = result_dir_path / "base_results.json"
    summary_path = result_dir_path / "comparison_summary.json"
    missing_files = [str(path) for path in [ft_path, base_path, summary_path] if not path.exists()]
    if missing_files:
        raise FileNotFoundError(f"标准化评估结果缺失: {missing_files}")

    with open(ft_path, "r", encoding="utf-8") as f:
        ft_results = json.load(f)
    with open(base_path, "r", encoding="utf-8") as f:
        base_results = json.load(f)
    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    finetuned_metrics = summary.get("finetuned") or aggregate_metric_dicts([item.get("metrics", {}) for item in ft_results])
    base_metrics = summary.get("base") or aggregate_metric_dicts([item.get("metrics", {}) for item in base_results])
    return {
        "result_dir": str(result_dir_path),
        "split_name": POST_TRAIN_EVAL_SPLIT_NAME,
        "finetuned_count": len(ft_results),
        "base_count": len(base_results),
        "finetuned_metrics": finetuned_metrics,
        "base_metrics": base_metrics,
        "gpu_stats": summary.get("gpu_stats", {}),
        "load_balance_reports": summary.get("load_balance_reports", {}),
        "report_files": summary.get("load_balance_report_files", {}),
        "files": {
            "finetuned_results": str(ft_path),
            "base_results": str(base_path),
            "comparison_summary": str(summary_path),
        },
    }


def print_metric_summary(title: str, metrics: Dict[str, Any]):
    print(f"\n{title}")
    print("-" * 60)
    if not metrics:
        print("  暂无指标")
        return
    preferred_keys = [
        "mean_precision",
        "mean_recall",
        "mean_f1",
        "mean_mean_match_iou",
        "success_rate",
        "total_samples",
    ]
    for key in preferred_keys:
        if key in metrics:
            value = metrics[key]
            if isinstance(value, float):
                print(f"  {key}: {value:.4f}")
            else:
                print(f"  {key}: {value}")


def print_post_training_eval_summary(summary: Dict[str, Any], stage_name: str):
    print("\n" + "=" * 72)
    print(stage_name)
    print("=" * 72)
    print(f"结果目录: {summary.get('result_dir')}")
    print(f"数据split: {summary.get('split_name')}")
    print(f"样本数: finetuned={summary.get('finetuned_count')} | base={summary.get('base_count')}")
    print_metric_summary("微调模型指标", summary.get("finetuned_metrics", {}))
    print_metric_summary("基础模型指标", summary.get("base_metrics", {}))

    gpu_stats = summary.get("gpu_stats", {})
    if gpu_stats:
        print("\nGPU统计摘要")
        print("-" * 60)
        for worker_name, stats in sorted(gpu_stats.items()):
            print(
                f"  {worker_name}: processed={stats.get('processed', 0)}, "
                f"failed={stats.get('failed', 0)}, "
                f"throughput={stats.get('throughput_samples_per_second', 0):.3f} sample/s"
            )

    load_balance_reports = summary.get("load_balance_reports", {})
    if load_balance_reports:
        print("\n调度性能对比")
        print("-" * 60)
        for model_name, report in sorted(load_balance_reports.items()):
            print(
                f"  {model_name}: scheduler={report.get('scheduler_mode', 'unknown')}, "
                f"delta={report.get('makespan_delta_seconds_vs_static', 0):.3f}s, "
                f"improvement={report.get('improvement_pct_vs_static', 0):.2f}%"
            )


def maybe_run_standardized_eval(
    lora_adapter_path: str,
    stage_name: str,
    allow_multi_gpu: bool = True,
    force_rerun: bool = False,
) -> Dict[str, Any]:
    global POST_TRAIN_EVAL_SUMMARY, POST_TRAIN_EVAL_FILES, POST_TRAIN_EVAL_LAST_RETURN_CODE

    if not POST_TRAIN_EVAL_ENABLED:
        print("标准化评估已禁用")
        return {}
    if not POST_TRAIN_EVAL_DATA_PATH:
        print("未发现评估数据文件, 跳过标准化评估")
        return {}
    if not POST_TRAIN_EVAL_GPU_IDS:
        print("未检测到可用GPU, 跳过标准化评估")
        return {}

    adapter_dir = Path(lora_adapter_path)
    if not adapter_dir.exists():
        print(f"LoRA目录不存在, 跳过标准化评估: {adapter_dir}")
        return {}

    eval_mode = "single_gpu"
    eval_gpu_ids = POST_TRAIN_EVAL_GPU_IDS[:1]
    if allow_multi_gpu and TRAINING_MODE in MULTI_GPU_TRAINING_MODES and len(POST_TRAIN_EVAL_GPU_IDS) > 1:
        eval_mode = "multi_gpu"
        eval_gpu_ids = POST_TRAIN_EVAL_GPU_IDS

    result_dir = Path(adapter_dir) / "post_training_eval"
    config = PostTrainingEvaluationConfig(
        mode=eval_mode,
        gpu_ids=eval_gpu_ids,
        base_model_path=BASE_MODEL_PATH,
        lora_adapter_path=str(adapter_dir),
        data_path=POST_TRAIN_EVAL_DATA_PATH,
        result_dir=str(result_dir),
        worker_script_path=POST_TRAIN_EVAL_SCRIPT_PATH,
        workdir=POST_TRAIN_EVAL_WORKDIR,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        iou_threshold=POST_TRAIN_EVAL_IOU_THRESHOLD,
        batch_size=POST_TRAIN_EVAL_BATCH_SIZE,
        max_eval_samples=POST_TRAIN_EVAL_MAX_EVAL_SAMPLES,
        max_new_tokens=POST_TRAIN_EVAL_MAX_NEW_TOKENS,
        temperature=POST_TRAIN_EVAL_TEMPERATURE,
        top_p=POST_TRAIN_EVAL_TOP_P,
        scheduler_mode=POST_TRAIN_EVAL_SCHEDULER_MODE,
        partition_strategy=POST_TRAIN_EVAL_PARTITION_STRATEGY,
        export_labelme=POST_TRAIN_EVAL_EXPORT_LABELME,
        torchrun_port=POST_TRAIN_EVAL_TORCHRUN_PORT,
        prompt_format=PROMPT_FORMAT,
        coord_order=COORD_ORDER,
    )
    config.print_summary()

    summary_path = result_dir / "comparison_summary.json"
    should_run = force_rerun or POST_TRAIN_EVAL_AUTO_RUN or not summary_path.exists()
    if should_run:
        POST_TRAIN_EVAL_LAST_RETURN_CODE = run_post_training_evaluation(
            config,
            raise_on_error=POST_TRAIN_EVAL_RAISE_ON_ERROR,
        )
    else:
        print(f"检测到已有标准化评估结果, 直接复用: {summary_path}")

    POST_TRAIN_EVAL_SUMMARY = load_post_training_evaluation_outputs(str(result_dir))
    POST_TRAIN_EVAL_FILES = POST_TRAIN_EVAL_SUMMARY.get("files", {})
    print_post_training_eval_summary(POST_TRAIN_EVAL_SUMMARY, stage_name)
    return POST_TRAIN_EVAL_SUMMARY


if TRAINING_MODE == "compare":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过推理测试")
elif TRAINING_MODE == "single":
    if ENABLE_PROMPT_SMOKE_TEST:
        test_prompts = [
            "请解释什么是机器学习。",
            "什么是深度学习？",
            "请介绍一下神经网络。",
        ]

        print("推理测试结果:")
        print("=" * 50)
        for prompt in test_prompts:
            print(f"\n问题: {prompt}")
            response = generate_chat_response(
                model,
                tokenizer,
                prompt,
                max_new_tokens=128,
                temperature=1.5,
                min_p=0.1,
            )
            print(f"回答: {response}")
            print("-" * 50)
    else:
        print("已禁用单GPU prompt smoke test")

    if POST_TRAIN_EVAL_ENABLED:
        print("\n提示: 标准化评估依赖磁盘LoRA目录，将在 Section 9 保存并重载模型后执行。")
else:
    # 多GPU模式: 训练在 Section 10 执行, 此处跳过评估
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过 Section 9 的评估")
    print(f"   多GPU训练将在 Section 10 执行, 训练完成后会自动保存 LoRA")
    print(f"   请在 Section 10 完成训练后, 再执行标准化评估")

⏭ 当前模式: DDP, 跳过 Section 9 的评估
   多GPU训练将在 Section 10 执行, 训练完成后会自动保存 LoRA
   请在 Section 10 完成训练后, 再执行标准化评估


## 8. 保存模型

统一保存 LoRA adapter、processor、训练摘要与 checkpoint 状态，便于断点续训与后续 Notebook 复用。

In [14]:
if TRAINING_MODE == "compare":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过保存LoRA")
else:
    from unsloth_finetune.training.distributed.adapter_utils import normalize_saved_adapter_config

    def _checkpoint_sort_key(path_obj: Path):
        suffix = path_obj.name.split("-")[-1]
        return int(suffix) if suffix.isdigit() else path_obj.name

    def collect_saved_artifact_summary(output_dir: str) -> Dict[str, Any]:
        output_path = Path(output_dir)
        checkpoint_dirs = sorted(
            [path for path in output_path.glob("checkpoint-*") if path.is_dir()],
            key=_checkpoint_sort_key,
        )
        trainer_state_files = sorted(
            str(path.relative_to(output_path))
            for path in output_path.glob("**/trainer_state.json")
            if path.is_file()
        )
        optimizer_state_files = sorted(
            str(path.relative_to(output_path))
            for path in output_path.glob("checkpoint-*/optimizer.pt")
            if path.is_file()
        )
        processor_files = sorted(
            path.name
            for path in output_path.iterdir()
            if path.is_file() and ("tokenizer" in path.name or "processor" in path.name or path.name.endswith(".json"))
        )
        adapter_files = sorted(
            path.name
            for path in output_path.iterdir()
            if path.is_file() and path.name.startswith("adapter_model")
        )
        return {
            "output_dir": str(output_path),
            "checkpoint_dirs": [path.name for path in checkpoint_dirs],
            "latest_checkpoint": str(checkpoint_dirs[-1]) if checkpoint_dirs else None,
            "trainer_state_files": trainer_state_files,
            "optimizer_state_files": optimizer_state_files,
            "processor_files": processor_files,
            "adapter_files": adapter_files,
            "resume_ready": bool(checkpoint_dirs and optimizer_state_files),
        }

    save_output_dir = Path(LORA_OUTPUT_DIR if TRAINING_MODE == "single" else resolve_latest_output_dir(LORA_OUTPUT_BASE, TRAINING_MODE, LORA_OUTPUT_DIR))
    save_output_dir.mkdir(parents=True, exist_ok=True)
    latest_path = Path(LORA_OUTPUT_BASE) / MODE_SUBDIR / "latest.txt"
    latest_path.parent.mkdir(parents=True, exist_ok=True)
    result_path = save_output_dir / "training_result.json"
    manifest_path = save_output_dir / "artifact_manifest.json"

    normalized_modules = []
    if TRAINING_MODE == "single":
        trainer.save_state()
        model.save_pretrained(str(save_output_dir))
        processor.save_pretrained(str(save_output_dir))
        try:
            normalized_modules = normalize_saved_adapter_config(str(save_output_dir))
        except FileNotFoundError as exc:
            print(f"LoRA配置规范化跳过: {exc}")

        latest_path.write_text(save_output_dir.name, encoding="utf-8")

        peak_vram = torch.cuda.max_memory_reserved() / 1024**3
        total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        training_result_payload = {
            "distributed_mode": "single_gpu",
            "world_size": 1,
            "learning_rate": _lr,
            "per_device_batch_size": _batch_size,
            "gradient_accumulation_steps": _grad_accum,
            "effective_batch_size": _batch_size * _grad_accum,
            "num_epochs": NUM_TRAIN_EPOCHS,
            "max_seq_length": MAX_SEQ_LENGTH,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "image_width": IMAGE_WIDTH,
            "image_height": IMAGE_HEIGHT,
            "train_loss": metrics.get("train_loss", 0),
            "train_runtime_sec": metrics.get("train_runtime", 0),
            "samples_per_second": metrics.get("train_samples_per_second", 0),
            "steps_per_second": metrics.get("train_steps_per_second", 0),
            "peak_vram_gb": round(peak_vram, 2),
            "total_vram_gb": round(total_vram, 1),
            "vram_utilization_pct": round(peak_vram / total_vram * 100, 1),
            "training_output_dir": str(save_output_dir),
        }
    else:
        if result_path.exists():
            with open(result_path, "r", encoding="utf-8") as f:
                training_result_payload = json.load(f)
        else:
            training_result_payload = {
                "distributed_mode": TRAINING_MODE,
                "world_size": len(POST_TRAIN_EVAL_GPU_IDS),
                "training_output_dir": str(save_output_dir),
            }
        try:
            normalized_modules = normalize_saved_adapter_config(str(save_output_dir))
        except FileNotFoundError as exc:
            print(f"LoRA配置规范化跳过: {exc}")
        latest_path.write_text(save_output_dir.name, encoding="utf-8")
        print(f"检测到分布式训练输出目录: {save_output_dir}")

    artifact_summary = collect_saved_artifact_summary(str(save_output_dir))
    artifact_summary.update(
        {
            "training_mode": TRAINING_MODE,
            "normalized_target_modules": normalized_modules,
            "normalized_target_modules_count": len(normalized_modules),
            "latest_marker": str(latest_path),
            "evaluation_result_dir": str(save_output_dir / "post_training_eval"),
        }
    )
    training_result_payload["artifact_summary"] = artifact_summary
    training_result_payload["saved_adapter_dir"] = str(save_output_dir)

    with open(result_path, "w", encoding="utf-8") as f:
        json.dump(training_result_payload, f, indent=2, ensure_ascii=False)
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(artifact_summary, f, indent=2, ensure_ascii=False)

    print(f"LoRA adapters目录: {save_output_dir}")
    print(f"latest标记已写入: {latest_path}")
    print(f"训练结果已保存: {result_path}")
    print(f"产物清单已保存: {manifest_path}")
    print(f"checkpoint目录: {artifact_summary['checkpoint_dirs']}")
    print(f"resume_ready: {artifact_summary['resume_ready']}")
    print("后续Notebook可通过 resolve_latest_output_dir(LORA_OUTPUT_BASE, TRAINING_MODE, LORA_OUTPUT_DIR) 引用最新模型")


检测到分布式训练输出目录: /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora/ddp_8gpu/20260521_073055
LoRA adapters目录: /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora/ddp_8gpu/20260521_073055
latest标记已写入: /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora/ddp_8gpu/latest.txt
训练结果已保存: /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora/ddp_8gpu/20260521_073055/training_result.json
产物清单已保存: /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora/ddp_8gpu/20260521_073055/artifact_manifest.json
checkpoint目录: []
resume_ready: False
后续Notebook可通过 resolve_latest_output_dir(LORA_OUTPUT_BASE, TRAINING_MODE, LORA_OUTPUT_DIR) 引用最新模型


In [15]:
if TRAINING_MODE != "single":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过保存GGUF")
else:
    # 选项：保存为GGUF格式（用于Ollama部署）
    # 取消注释以执行

    # GGUF_OUTPUT_DIR = str(PROJECT_ROOT / 'models' / 'finetuned' / 'gemma4_e4b_gguf')
    # QUANTIZATION_METHOD = "q4_k_m"  # 可选: q4_k_m, q8_0, f16

    # model.save_pretrained_gguf(
    #     GGUF_OUTPUT_DIR,
    #     tokenizer,
    #     quantization_method=QUANTIZATION_METHOD,
    # )

    # print(f"GGUF模型已保存到: {GGUF_OUTPUT_DIR}")
    # print(f"量化方法: {QUANTIZATION_METHOD}")
    pass

⏭ 当前模式: DDP, 跳过保存GGUF


In [16]:
if TRAINING_MODE != "single":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过推送HF Hub")
else:
    # 选项：推送到Hugging Face Hub
    # 取消注释以执行（需要设置HF_TOKEN）

    # from huggingface_hub import login
    # login(token="YOUR_HF_TOKEN")  # 替换为您的HF token

    # HF_REPO_NAME = "your-username/gemma4-e4b-finetuned"  # 替换为您的repo名称
    # model.push_to_hub(HF_REPO_NAME)
    # tokenizer.push_to_hub(HF_REPO_NAME)

    # print(f"模型已推送到: https://huggingface.co/{HF_REPO_NAME}")
    pass

⏭ 当前模式: DDP, 跳过推送HF Hub


## 9. 加载保存的模型进行测试

验证保存的权重可正常加载，并在需要时复用或重新生成标准化评估结果。

In [17]:
if TRAINING_MODE != "single":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过模型加载测试（多GPU模式训练结果已保存在LoRA目录中）")
else:
    os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

    def patch_peft_for_gemma4():
        """
        修复PEFT对Gemma4ClippableLinear的支持。
        使用类名字符串匹配代替isinstance, 遞免Unsloth monkey-patch导致的类身份不匹配。
        """
        try:
            from peft.tuners.lora import model as lora_model

            original_create_new_module = lora_model.LoraModel._create_new_module

            def patched_create_new_module(lora_config, adapter_name, target, **kwargs):
                if target.__class__.__name__ == "Gemma4ClippableLinear" and hasattr(target, "linear"):
                    return original_create_new_module(lora_config, adapter_name, target.linear, **kwargs)
                return original_create_new_module(lora_config, adapter_name, target, **kwargs)

            lora_model.LoraModel._create_new_module = staticmethod(patched_create_new_module)
            print("PEFT已patch，支持Gemma4ClippableLinear")
            return True
        except Exception as exc:
            print(f"Patch失败: {exc}")
            print("将使用exclude_modules方式加载")
            return False

    patch_success = patch_peft_for_gemma4()
    load_dir = resolve_latest_output_dir(LORA_OUTPUT_BASE, TRAINING_MODE, LORA_OUTPUT_DIR)
    print(f"\n将加载模型路径: {load_dir}")
    if Path(load_dir).exists():
        print(f"\n从磁盘加载保存的模型: {load_dir}")
        print("正在加载基础模型...")
        load_kwargs = {
            "model_name": BASE_MODEL_PATH,
            "max_seq_length": MAX_SEQ_LENGTH,
            "load_in_4bit": LOAD_IN_4BIT,
            "disable_log_stats": True,
        }
        if torch.cuda.is_available():
            load_kwargs["device_map"] = {"": 0}
        loaded_base_model, loaded_processor = FastVisionModel.from_pretrained(**load_kwargs)
        loaded_tokenizer = getattr(loaded_processor, "tokenizer", loaded_processor)

        print("正在加载LoRA adapter...")
        from peft import PeftModel

        try:
            loaded_model = PeftModel.from_pretrained(loaded_base_model, load_dir, is_trainable=False)
            print("LoRA adapter加载成功！")
        except (ValueError, TypeError) as exc:
            print(f"直接加载失败，使用exclude_modules方式: {exc}")
            from peft import LoraConfig
            from safetensors.torch import load_file

            adapter_config = LoraConfig.from_pretrained(load_dir)
            adapter_config.exclude_modules = ["vision_tower", "audio_tower"]
            loaded_model = PeftModel(loaded_base_model, adapter_config, adapter_name="default")
            adapter_weights_path = Path(load_dir) / "adapter_model.safetensors"
            if adapter_weights_path.exists():
                state_dict = load_file(str(adapter_weights_path), device="cpu")
                loaded_model.load_state_dict(state_dict, strict=False)
            print("LoRA adapter权重手动加载完成")

        print("\n模型加载完成！开始测试...")
        test_prompts_reload = [
            "请解释什么是机器学习。",
            "什么是深度学习？",
        ]

        print("\n重新加载后的模型测试结果:")
        print("=" * 60)
        for prompt in test_prompts_reload:
            print(f"\n问题: {prompt}")
            response = generate_chat_response(
                loaded_model,
                loaded_tokenizer,
                prompt,
                max_new_tokens=128,
                temperature=0.7,
                top_p=0.9,
            )
            print(f"回答: {response}")
            print("-" * 60)

        print(f"\nLoRA adapters目录内容: {load_dir}")
        saved_files = sorted(path.name for path in Path(load_dir).iterdir())
        print(f"保存的文件: {saved_files}")

        allow_multi_gpu_eval = TRAINING_MODE in MULTI_GPU_TRAINING_MODES
        maybe_run_standardized_eval(
            load_dir,
            stage_name="保存模型加载后标准化评估结果",
            allow_multi_gpu=allow_multi_gpu_eval,
            force_rerun=False,
        )
    else:
        print(f"模型目录不存在: {load_dir}")
        print("请先运行前面的训练和保存步骤")
        print(f"PEFT patch状态: {'成功' if patch_success else '失败'}")

⏭ 当前模式: DDP, 跳过模型加载测试（多GPU模式训练结果已保存在LoRA目录中）


---

# 附录：8×A6000 多GPU分布式训练优化

本节针对 **8张 NVIDIA A6000 GPU (48GB VRAM)** 服务器环境，提供完整的多GPU微调优化方案。

## 8×A6000 优化策略总览

| 优化维度   | 单GPU配置 | 8×A6000优化配置              | 优化原理                           |
| ---------- | --------- | ---------------------------- | ---------------------------------- |
| 分布式框架 | 无        | **DDP** (推荐)               | QLoRA E4B可放入单卡，DDP通信开销低 |
| 混合精度   | FP32/FP16 | **BF16**                     | A6000 Ampere架构原生支持           |
| 每GPU批次  | 2         | **4**                        | 48GB VRAM充足，QLoRA仅~10GB        |
| 梯度累积   | 4         | **2**                        | 有效batch=4×2×8=64                 |
| 学习率     | 2e-5      | **4e-5×8=3.2e-4** (线性缩放) | 多GPU需同步梯度，增大LR补偿        |
| GPU监控    | 无        | **实时监控+CSV日志**         | 监控显存/利用率/温度               |

## DDP vs FSDP 选择

| 方式    | 适用场景                 | 显存优势           | 通信开销 | 推荐度            |
| ------- | ------------------------ | ------------------ | -------- | ----------------- |
| **DDP** | 单机多卡，模型可放入单卡 | 无额外优势         | 较低     | ⭐⭐⭐ (E4B推荐)  |
| FSDP    | 大模型(31B+)，多机多卡   | 显存分片，大幅节省 | 较高     | ⭐⭐ (大模型推荐) |

## 10. 多GPU训练配置

分布式训练需要将Notebook转换为独立的Python脚本，使用 `torchrun` 或 `accelerate launch` 启动。


### 10.1 GPU环境检测与配置

使用 `gpu_monitor.py` 模块检测多GPU环境，包括显存容量、BF16支持、NVLink拓扑等信息。


In [18]:
# 8×A6000 GPU环境检测
# ============================================================
# 使用 gpu_monitor 模块进行全面GPU环境检测

import sys

sys.path.insert(0, str(PROJECT_ROOT))

from unsloth_finetune.training.distributed.gpu_monitor import print_gpu_info

# 打印详细GPU信息
print_gpu_info()

# GPU监控器由 train_distributed.py 在训练时创建
# 这里只做环境检测，不创建监控器实例
num_gpus = torch.cuda.device_count()
if num_gpus < 2:
    print(f"\n警告: 检测到 {num_gpus} GPU, 分布式训练需要 >= 2 GPU")
    print("请在多GPU服务器上运行")

检测到 8 张GPU:
 GPU |                             名称 |    总显存(GB) |       CUDA计算能力 |   BF16支持
--------------------------------------------------------------------------------
   0 | NVIDIA RTX 5880 Ada Generation |       47.4 | 8.            9 |        ✓
   1 | NVIDIA RTX 5880 Ada Generation |       47.4 | 8.            9 |        ✓
   2 | NVIDIA RTX 5880 Ada Generation |       47.4 | 8.            9 |        ✓
   3 | NVIDIA RTX 5880 Ada Generation |       47.4 | 8.            9 |        ✓
   4 | NVIDIA RTX 5880 Ada Generation |       47.4 | 8.            9 |        ✓
   5 | NVIDIA RTX 5880 Ada Generation |       47.4 | 8.            9 |        ✓
   6 | NVIDIA RTX 5880 Ada Generation |       47.4 | 8.            9 |        ✓
   7 | NVIDIA RTX 5880 Ada Generation |       47.4 | 8.            9 |        ✓

BF16混合精度: 可用
CUDA版本: 12.8
PyTorch版本: 2.10.0+cu128
NCCL可用: 是


### 10.2 训练配置优化计算

根据GPU数量、显存容量和模型大小，计算最优的批处理大小、梯度累积步数和学习率。


In [19]:
if TRAINING_MODE not in ("DDP", "FSDP"):
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过分布式配置优化计算")
else:
    # 8×A6000 训练配置优化计算
    # ============================================================
    # 根据GPU数量、显存容量和模型参数量，计算最优配置

    # 显存估算 (QLoRA E4B in 4-bit)
    model_params_b = 4.0  # E4B模型参数量 (Billion)
    qlora_vram_per_gb = 0.5  # 4-bit量化: 每B参数约0.5GB
    model_base_vram = model_params_b * qlora_vram_per_gb  # 基础模型显存
    optimizer_vram = 1.5  # 8-bit优化器显存
    activation_overhead = 2.0  # 激活值/梯度显存 (与batch_size成正比)
    safety_margin = 0.85  # 显存安全系数 (不超过85%)

    gpu_vram_gb = 48.0  # A6000每卡48GB
    n_gpus = torch.cuda.device_count()

    # 单GPU可承受的最大batch_size
    available_vram = gpu_vram_gb * safety_margin - model_base_vram - optimizer_vram
    max_batch_size = int(available_vram / (activation_overhead / 2))  # 每样本约1GB

    # 计算最优配置
    if n_gpus >= 8:
        optimal_batch = min(MULTI_GPU_BATCH_SIZE, max_batch_size)
        optimal_grad_accum = MULTI_GPU_GRAD_ACCUM
        effective_batch = optimal_batch * optimal_grad_accum * n_gpus
    elif n_gpus >= 2:
        optimal_batch = min(4, max_batch_size)
        optimal_grad_accum = 2
        effective_batch = optimal_batch * optimal_grad_accum * n_gpus
    else:
        optimal_batch = PER_DEVICE_TRAIN_BATCH_SIZE
        optimal_grad_accum = GRADIENT_ACCUMULATION_STEPS
        effective_batch = optimal_batch * optimal_grad_accum

    # 学习率缩放计算
    if MULTI_GPU_ENABLED and n_gpus >= 2:
        if MULTI_GPU_LR_SCALING == "linear":
            scaled_lr = MULTI_GPU_LR_BASE * n_gpus
        elif MULTI_GPU_LR_SCALING == "sqrt":
            scaled_lr = MULTI_GPU_LR_BASE * (n_gpus**0.5)
        else:
            scaled_lr = MULTI_GPU_LR_BASE
    else:
        scaled_lr = LEARNING_RATE

    # 打印优化配置结果
    print("=" * 60)
    print(f"训练配置优化计算结果")
    print("=" * 60)
    print(f"\n显存估算:")
    print(f"  模型基础显存: {model_base_vram:.1f}GB")
    print(f"  优化器显存: {optimizer_vram:.1f}GB")
    print(f"  可用显存(85%安全系数): {available_vram:.1f}GB")
    print(f"  最大batch_size: {max_batch_size}")
    print(f"\n最优配置 ({n_gpus} GPU):")
    print(f"  每GPU batch_size: {optimal_batch}")
    print(f"  gradient_accumulation: {optimal_grad_accum}")
    print(f"  有效全局批次: {effective_batch}")
    print(f"  学习率: {scaled_lr:.6f} (缩放策略: {MULTI_GPU_LR_SCALING})")
    print(f"  混合精度: BF16={MULTI_GPU_BF16}")
    print(f"\n预期性能提升:")
    if n_gpus >= 2:
        ideal_speedup = n_gpus
        practical_speedup = ideal_speedup * 0.85  # 通信开销约15%
        print(f"  理想加速比: {ideal_speedup:.0f}x")
        print(f"  实际加速比: ~{practical_speedup:.1f}x")
        print(f"  显存利用率提升: 从 {model_base_vram/48*100:.1f}% 到 ~{available_vram/48*100:.1f}%")

训练配置优化计算结果

显存估算:
  模型基础显存: 2.0GB
  优化器显存: 1.5GB
  可用显存(85%安全系数): 37.3GB
  最大batch_size: 37

最优配置 (8 GPU):
  每GPU batch_size: 8
  gradient_accumulation: 2
  有效全局批次: 128
  学习率: 0.000320 (缩放策略: linear)
  混合精度: BF16=True

预期性能提升:
  理想加速比: 8x
  实际加速比: ~6.8x
  显存利用率提升: 从 4.2% 到 ~77.7%


### 10.3 8×A6000 DDP 启动命令

使用 `!{变量名}` 语法可直接在Notebook中执行终端命令，无需复制到终端。取消注释 `!` 行即可直接执行。

⚠️ DDP需要多进程(torchrun)，Notebook单进程执行可能受限。如果执行失败，请在终端中直接运行。

所有参数已针对A6000 48GB优化。


In [20]:
if TRAINING_MODE not in ("DDP", "device_map", "FSDP", "auto", "multi_node"):
    print(f"⏭ 当前模式: {TRAINING_MODE}, 不执行分布式训练命令")
else:
    # === CUDA 环境清理 ===
    # 在 Notebook 内启动 torchrun 前彻底释放 kernel 持有的 CUDA 资源,
    # 避免 Jupyter CUDA context 与 DDP 子进程冲突导致间歇性挂死
    import gc as _gc
    # 1. 释放所有 CUDA tensor -> CPU (断开 GPU 引用)
    _cuda_tensors = []
    for _obj in _gc.get_objects():
        if torch.is_tensor(_obj) and _obj.is_cuda:
            _cuda_tensors.append(_obj.detach_().cpu())
    del _cuda_tensors
    _gc.collect()
    # 2. 清空 CUDA 缓存分配器 + 重置峰值统计
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    # 3. NCCL 超时保护: 10分钟超时 + 异步错误检测
    # 防止 NCCL barrier 无限等待 (默认30分钟太长)
    # 防止进程挂死时永远阻塞 (async error -> 报错而非挂死)
    os.environ["NCCL_ASYNC_ERROR_HANDLING"] = "1"
    os.environ.setdefault("NCCL_TIMEOUT", "600")
    os.environ.setdefault("NCCL_DEBUG", "WARN")
    print("CUDA 清理完成 + NCCL 超时保护已设置 (timeout=600s, async_error=on)")

    n_gpus = torch.cuda.device_count()
    import json

    runtime_args = [
        f"--image_width {IMAGE_WIDTH}",
        f"--image_height {IMAGE_HEIGHT}",
        f"--image_load_mode {IMAGE_LOAD_MODE}",
    ]
    if TF32_ENABLED:
        runtime_args.append("--tf32")
    if CPU_THREADS_PER_RANK is not None:
        runtime_args.append(f"--cpu_threads_per_rank {CPU_THREADS_PER_RANK}")
    if DATALOADER_NUM_WORKERS is not None:
        runtime_args.append(f"--dataloader_num_workers {DATALOADER_NUM_WORKERS}")
    if DATALOADER_PREFETCH_FACTOR is not None:
        runtime_args.append(f"--dataloader_prefetch_factor {DATALOADER_PREFETCH_FACTOR}")
    runtime_args.append("--dataloader_pin_memory" if DATALOADER_PIN_MEMORY else "--no-dataloader_pin_memory")
    runtime_args.append("--dataloader_persistent_workers" if DATALOADER_PERSISTENT_WORKERS else "--no-dataloader_persistent_workers")
    runtime_args.append("--dataloader_drop_last" if DATALOADER_DROP_LAST else "--no-dataloader_drop_last")
    if IMAGE_BATCH_SIZE is not None:
        runtime_args.append(f"--image_batch_size {IMAGE_BATCH_SIZE}")
    if MATERIALIZE_VISION_DATASET:
        runtime_args.append("--materialize_vision_dataset")
    runtime_args_str = "".join(f" {arg}" for arg in runtime_args)

    if TRAINING_MODE == "DDP":
        _log_dir = DDP_2X_GPU_LOG_DIR if MODELS_PER_GPU > 1 else DDP_GPU_LOG_DIR
        _mode_desc = f"DDP 8卡{' + ' + str(MODELS_PER_GPU) + '倍吞吐' if MODELS_PER_GPU > 1 else ''}"
        ddp_cmd = (
            f"PYTHONPATH={str(PROJECT_ROOT)} torchrun --nproc_per_node={n_gpus} {TRAIN_SCRIPT_PATH}"
            f" --model_name {BASE_MODEL_PATH}"
            f" --data_path {ACTUAL_DATA_PATH}"
            f" --output_dir {LORA_OUTPUT_DIR}"
            f" --max_seq_length {MAX_SEQ_LENGTH}"
            f" --per_device_batch_size {MULTI_GPU_BATCH_SIZE}"
            f" --models_per_gpu {MODELS_PER_GPU}"
            f" --gradient_accumulation_steps {MULTI_GPU_GRAD_ACCUM}"
            f" --learning_rate {MULTI_GPU_LR_BASE}"
            f" --lr_scaling {MULTI_GPU_LR_SCALING}"
            f" --warmup_ratio {MULTI_GPU_WARMUP_RATIO}"
            f" --num_epochs {NUM_TRAIN_EPOCHS}"
            " --bf16"
            " --vision_mode"
            " --use_ddp"
            " --gpu_monitor"
            f" --gpu_log_dir {_log_dir}"
            f" --gpu_log_interval {MULTI_GPU_LOG_INTERVAL}"
            f"{runtime_args_str}"
        )
        print(f"{_mode_desc}训练命令:")
        print(ddp_cmd)
        print(f"输出目录: {LORA_OUTPUT_DIR} (自动按模式/时间戳隔离)")
        print(f"训练图片尺寸: {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
        if MODELS_PER_GPU > 1:
            print(f"有效批次: {MULTI_GPU_BATCH_SIZE * MULTI_GPU_GRAD_ACCUM * n_gpus * MODELS_PER_GPU} ({n_gpus}卡 × {MODELS_PER_GPU}倍吞吐)")
        print("\n提示: Notebook中执行torchrun可能受限，建议在终端中运行上述命令")
        print("执行方式: 复制上述命令到终端执行，或取消注释下面行")
        _train_exit_code = get_ipython().system(ddp_cmd)

        if _train_exit_code == 0:
            latest_path = Path(LORA_OUTPUT_BASE) / MODE_SUBDIR / "latest.txt"
            latest_path.parent.mkdir(parents=True, exist_ok=True)
            latest_path.write_text(TRAIN_TIMESTAMP)
            print(f"latest标记已写入: {latest_path}")
        else:
            print(f"训练命令退出码: {_train_exit_code}，跳过写入latest.txt")

    elif TRAINING_MODE == "device_map":
        _num_groups = len(GPU_GROUPS)
        devicemap_cmd = (
            f"PYTHONPATH={str(PROJECT_ROOT)} torchrun --nproc_per_node={_num_groups} {TRAIN_SCRIPT_PATH}"
            f" --model_name {BASE_MODEL_PATH}"
            f" --data_path {ACTUAL_DATA_PATH}"
            f" --output_dir {LORA_OUTPUT_DIR}"
            f" --max_seq_length {MAX_SEQ_LENGTH}"
            f" --per_device_batch_size {MULTI_GPU_BATCH_SIZE}"
            f" --gradient_accumulation_steps {MULTI_GPU_GRAD_ACCUM}"
            f" --learning_rate {MULTI_GPU_LR_BASE}"
            f" --lr_scaling {MULTI_GPU_LR_SCALING}"
            f" --warmup_ratio {MULTI_GPU_WARMUP_RATIO}"
            f" --num_epochs {NUM_TRAIN_EPOCHS}"
            " --bf16"
            " --vision_mode"
            " --use_ddp"
            f" --device_map {DEVICE_MAP_STRATEGY}"
            f" --gpu_groups '{json.dumps(GPU_GROUPS)}'"
            " --gpu_monitor"
            f" --gpu_log_dir {DEVICEMAP_GPU_LOG_DIR}"
            f" --gpu_log_interval {MULTI_GPU_LOG_INTERVAL}"
            f"{runtime_args_str}"
        )
        print("device_map 2D并行训练命令 (8卡分4组):")
        print(devicemap_cmd)
        print(f"输出目录: {LORA_OUTPUT_DIR} (自动按模式/时间戳隔离)")
        print(f"GPU分组: {GPU_GROUPS} → {_num_groups}路数据并行 × {len(GPU_GROUPS[0])}卡模型并行")
        print(f"训练图片尺寸: {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
        print("\n提示: device_map适合大模型场景，建议在终端中运行上述命令")
        _train_exit_code = get_ipython().system(devicemap_cmd)

        if _train_exit_code == 0:
            latest_path = Path(LORA_OUTPUT_BASE) / MODE_SUBDIR / "latest.txt"
            latest_path.parent.mkdir(parents=True, exist_ok=True)
            latest_path.write_text(TRAIN_TIMESTAMP)
            print(f"latest标记已写入: {latest_path}")
        else:
            print(f"训练命令退出码: {_train_exit_code}，跳过写入latest.txt")

    elif TRAINING_MODE == "FSDP":
        fsdp_cmd = (
            f"PYTHONPATH={str(PROJECT_ROOT)} torchrun --nproc_per_node={n_gpus} {TRAIN_SCRIPT_PATH}"
            f" --model_name {BASE_MODEL_PATH}"
            f" --data_path {ACTUAL_DATA_PATH}"
            f" --output_dir {LORA_OUTPUT_DIR}"
            f" --max_seq_length {MAX_SEQ_LENGTH}"
            f" --per_device_batch_size {MULTI_GPU_BATCH_SIZE}"
            f" --gradient_accumulation_steps {MULTI_GPU_GRAD_ACCUM}"
            f" --learning_rate {MULTI_GPU_LR_BASE}"
            f" --lr_scaling {MULTI_GPU_LR_SCALING}"
            f" --warmup_ratio {MULTI_GPU_WARMUP_RATIO}"
            f" --num_epochs {NUM_TRAIN_EPOCHS}"
            " --bf16"
            " --vision_mode"
            " --use_fsdp"
            " --gpu_monitor"
            f" --gpu_log_dir {FSDP_GPU_LOG_DIR}"
            f" --gpu_log_interval {MULTI_GPU_LOG_INTERVAL}"
            f"{runtime_args_str}"
        )
        print("FSDP 8卡训练命令:")
        print(fsdp_cmd)
        print(f"输出目录: {LORA_OUTPUT_DIR} (自动按模式/时间戳隔离)")
        print(f"训练图片尺寸: {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
        print("\n提示: FSDP适合更大模型(如31B)，建议在终端中运行上述命令")
        _train_exit_code = get_ipython().system(fsdp_cmd)

        if _train_exit_code == 0:
            latest_path = Path(LORA_OUTPUT_BASE) / MODE_SUBDIR / "latest.txt"
            latest_path.parent.mkdir(parents=True, exist_ok=True)
            latest_path.write_text(TRAIN_TIMESTAMP)
            print(f"latest标记已写入: {latest_path}")
        else:
            print(f"训练命令退出码: {_train_exit_code}，跳过写入latest.txt")

    elif TRAINING_MODE == "auto":
        auto_cmd = (
            f"PYTHONPATH={str(PROJECT_ROOT)} torchrun --nproc_per_node={n_gpus} {TRAIN_SCRIPT_PATH}"
            f" --model_name {BASE_MODEL_PATH}"
            f" --data_path {ACTUAL_DATA_PATH}"
            f" --output_dir {LORA_OUTPUT_DIR}"
            f" --auto_detect"
            f" --model_vram_gb {MODEL_VRAM_GB}"
            f" --per_device_batch_size {MULTI_GPU_BATCH_SIZE}"
            f" --gradient_accumulation_steps {MULTI_GPU_GRAD_ACCUM}"
            f" --learning_rate {MULTI_GPU_LR_BASE}"
            " --bf16"
            " --vision_mode"
            " --gpu_monitor"
            f" --gpu_log_dir {AUTO_GPU_LOG_DIR}"
            f"{runtime_args_str}"
        )
        print("自动检测模式训练命令:")
        print(auto_cmd)
        print(f"输出目录: {LORA_OUTPUT_DIR} (自动按模式/时间戳隔离)")
        print(f"模型显存需求: {MODEL_VRAM_GB}GB")
        print(f"训练图片尺寸: {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
        print("\n提示: 自动检测会根据GPU显存自动选择最优模式 (DDP/device_map/FSDP)")
        _train_exit_code = get_ipython().system(auto_cmd)

        if _train_exit_code == 0:
            latest_path = Path(LORA_OUTPUT_BASE) / MODE_SUBDIR / "latest.txt"
            latest_path.parent.mkdir(parents=True, exist_ok=True)
            latest_path.write_text(TRAIN_TIMESTAMP)
            print(f"latest标记已写入: {latest_path}")
        else:
            print(f"训练命令退出码: {_train_exit_code}，跳过写入latest.txt")

    elif TRAINING_MODE == "multi_node":
        MASTER_ADDR = "192.168.1.1"
        MASTER_PORT = 29500
        NNODES = 2
        NODE_RANK = 0

        multi_node_cmd = (
            f"PYTHONPATH={str(PROJECT_ROOT)} torchrun --nnodes={NNODES} --nproc_per_node={n_gpus} --node_rank={NODE_RANK}"
            f" --master_addr={MASTER_ADDR} --master_port={MASTER_PORT}"
            f" {TRAIN_SCRIPT_PATH}"
            f" --model_name {BASE_MODEL_PATH}"
            f" --data_path {ACTUAL_DATA_PATH}"
            f" --output_dir {LORA_OUTPUT_DIR}"
            f" --max_seq_length {MAX_SEQ_LENGTH}"
            f" --per_device_batch_size {MULTI_GPU_BATCH_SIZE}"
            f" --gradient_accumulation_steps {MULTI_GPU_GRAD_ACCUM}"
            f" --learning_rate {MULTI_GPU_LR_BASE}"
            f" --lr_scaling {MULTI_GPU_LR_SCALING}"
            f" --warmup_ratio {MULTI_GPU_WARMUP_RATIO}"
            f" --num_epochs {NUM_TRAIN_EPOCHS}"
            " --bf16"
            " --vision_mode"
            " --use_ddp"
            " --gpu_monitor"
            f"{runtime_args_str}"
        )
        print("多机多卡训练命令:")
        print(multi_node_cmd)
        print(f"输出目录: {LORA_OUTPUT_DIR} (自动按模式/时间戳隔离)")
        print(f"训练图片尺寸: {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
        print(f"\n⚠️ 需要配置: MASTER_ADDR={MASTER_ADDR}, MASTER_PORT={MASTER_PORT}")
        print("在每个节点上执行相同命令，仅修改 NODE_RANK (0, 1, 2...)")
        _train_exit_code = get_ipython().system(multi_node_cmd)

        if _train_exit_code == 0:
            latest_path = Path(LORA_OUTPUT_BASE) / MODE_SUBDIR / "latest.txt"
            latest_path.parent.mkdir(parents=True, exist_ok=True)
            latest_path.write_text(TRAIN_TIMESTAMP)
            print(f"latest标记已写入: {latest_path}")
        else:
            print(f"训练命令退出码: {_train_exit_code}，跳过写入latest.txt")

    print(f"\n训练脚本路径: {TRAIN_SCRIPT_PATH}")
    print("详细配置请参考: docs/project-structure-guide.md")


CUDA 清理完成 + NCCL 超时保护已设置 (timeout=600s, async_error=on)
DDP 8卡训练命令:
PYTHONPATH=/raid5/sh/code/unsloth-finetune torchrun --nproc_per_node=8 /raid5/sh/code/unsloth-finetune/src/unsloth_finetune/training/distributed/train_distributed.py --model_name /raid5/sh/model/unsloth/gemma-4-E4B-it-unsloth-bnb-4bit --data_path /raid5/sh/code/unsloth-finetune/data/processed/unsloth_training_data-wgang_40/train.jsonl --output_dir /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora/ddp_8gpu/20260521_073055 --max_seq_length 2048 --per_device_batch_size 8 --models_per_gpu 1 --gradient_accumulation_steps 2 --learning_rate 0.00032 --lr_scaling linear --warmup_ratio 0.06 --num_epochs 1 --bf16 --vision_mode --use_ddp --gpu_monitor --gpu_log_dir /raid5/sh/code/unsloth-finetune/notebooks/gpu_logs/ddp_8gpu --gpu_log_interval 20 --image_width 512 --image_height 512 --image_load_mode lazy --tf32 --dataloader_pin_memory --dataloader_persistent_workers --no-dataloader_drop_last
输出目录: /raid5/sh/code/uns

In [21]:
# ============================================================
# 多GPU训练后标准化评估
# ============================================================
# 训练完成后，基于已保存的 LoRA adapter 执行标准化评估

if TRAINING_MODE not in MULTI_GPU_TRAINING_MODES:
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过多GPU训练后评估")
else:
    # 获取最新训练的 LoRA 目录
    eval_adapter_dir = resolve_latest_output_dir(LORA_OUTPUT_BASE, TRAINING_MODE, LORA_OUTPUT_DIR)
    print(f"多GPU训练完成，开始执行标准化评估: {eval_adapter_dir}")

    maybe_run_standardized_eval(
        eval_adapter_dir,
        stage_name="多GPU训练后标准化评估结果",
        allow_multi_gpu=True,
        force_rerun=False,
    )

# ============================================================
# 性能对比分析 (仅 compare 模式)
# ============================================================
# 对比单GPU和多GPU的评估结果

if TRAINING_MODE != "compare":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过性能对比分析")
else:
    print("开始性能对比分析...")
    # 对比逻辑在 Cell 38 中详细实现


多GPU训练完成，开始执行标准化评估: /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora/ddp_8gpu/20260521_073055
训练后标准化评估摘要
评估模式: multi_gpu
GPU IDs: [0, 1, 2, 3, 4, 5, 6, 7]
数据集: /raid5/sh/code/unsloth-finetune/data/processed/unsloth_training_data-wgang_40/test.jsonl
LoRA目录: /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora/ddp_8gpu/20260521_073055
结果目录: /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora/ddp_8gpu/20260521_073055/post_training_eval
worker脚本: /raid5/sh/code/unsloth-finetune/src/unsloth_finetune/training/distributed/distributed_inference.py
工作目录: /raid5/sh/code/unsloth-finetune
batch_size: 4
scheduler_mode: dynamic_queue
partition_strategy: round_robin
max_eval_samples: None
prompt_format: normalized_xyxy
coord_order: xyxy
自动执行: True

开始执行训练后标准化评估...
CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7
命令预览:
torchrun \
  --nproc_per_node=8 \
  --rdzv_backend=c10d \
  --rdzv_endpoint=localhost:29510 \
  /raid5/sh/code/unsloth-finetune/src/unsloth_finetune/trainin

W0521 07:39:03.813000 365720 site-packages/torch/distributed/run.py:852] 
W0521 07:39:03.813000 365720 site-packages/torch/distributed/run.py:852] *****************************************
W0521 07:39:03.813000 365720 site-packages/torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0521 07:39:03.813000 365720 site-packages/torch/distributed/run.py:852] *****************************************


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!


[W521 07:39:12.994931109 Utils.hpp:137] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())
[W521 07:39:12.014805136 Utils.hpp:137] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())


🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!


[W521 07:39:13.772735844 Utils.hpp:137] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())
[W521 07:39:13.809359846 Utils.hpp:137] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())
[W521 07:39:13.853855817 Utils.hpp:137] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())
[W521 07:39:13.863688636 Utils.hpp:137] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())
[W521 07:39:13.907483007 Utils.hpp:137] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use TORCH_NCCL_ASYNC_ERROR_HANDLING instead (function operator())
[W521 07:39:13.921058648 Utils.hpp:137] Warning: Environment variable NCCL_ASYNC_ERROR_HANDLING is deprecated; use 

NCCL version 2.27.5+cuda12.9


[2026-05-21 15:39:14 CST+0800] [rank 5] worker ready: local_rank=5, physical_gpu=5, world_size=8, total_samples=413, scheduler_mode=dynamic_queue, queue_batch_size=4
[2026-05-21 15:39:14 CST+0800] [rank 5] round 1 start: finetuned
[2026-05-21 15:39:14 CST+0800] [rank 1] worker ready: local_rank=1, physical_gpu=1, world_size=8, total_samples=413, scheduler_mode=dynamic_queue, queue_batch_size=4
[2026-05-21 15:39:14 CST+0800] [rank 1] round 1 start: finetuned
[2026-05-21 15:39:14 CST+0800] [rank 6] worker ready: local_rank=6, physical_gpu=6, world_size=8, total_samples=413, scheduler_mode=dynamic_queue, queue_batch_size=4
[2026-05-21 15:39:14 CST+0800] [rank 6] round 1 start: finetuned
[2026-05-21 15:39:14 CST+0800] [rank 7] worker ready: local_rank=7, physical_gpu=7, world_size=8, total_samples=413, scheduler_mode=dynamic_queue, queue_batch_size=4
[2026-05-21 15:39:14 CST+0800] [rank 7] round 1 start: finetuned
[2026-05-21 15:39:14 CST+0800] [rank 4] worker ready: local_rank=4, physical

评估进程已结束, return code = 0

多GPU训练后标准化评估结果
结果目录: /raid5/sh/code/unsloth-finetune/models/finetuned/gemma4_e4b_lora/ddp_8gpu/20260521_073055/post_training_eval
数据split: test
样本数: finetuned=413 | base=413

微调模型指标
------------------------------------------------------------
  mean_precision: 0.0847
  mean_recall: 0.0075
  mean_f1: 0.0077
  mean_mean_match_iou: 0.0058
  success_rate: 0.0073
  total_samples: 413

基础模型指标
------------------------------------------------------------
  mean_precision: 0.1822
  mean_recall: 0.0593
  mean_f1: 0.0524
  mean_mean_match_iou: 0.0371
  success_rate: 0.0460
  total_samples: 413

GPU统计摘要
------------------------------------------------------------
  base_rank0_gpu0: processed=52, failed=0, throughput=0.144 sample/s
  base_rank1_gpu1: processed=37, failed=0, throughput=0.105 sample/s
  base_rank2_gpu2: processed=56, failed=0, throughput=0.156 sample/s
  base_rank3_gpu3: processed=52, failed=0, throughput=0.145 sample/s
  base_rank4_gpu4: processed=60, faile

### 10.4 GPU监控集成与训练脚本

分布式训练脚本 `train_distributed.py` 已集成 `gpu_monitor.py` 模块，提供实时GPU监控和CSV日志。

GPU监控指标：

- 每GPU显存分配/预留/利用率/温度
- 训练速度（samples/s, steps/s）
- 峰值显存和显存效率

以下代码演示如何在训练中集成GPU监控。


In [22]:
if TRAINING_MODE != "single":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过GPU监控演示")
else:
    # GPU监控集成示例
    # ============================================================
    # 展示如何在单GPU训练中集成GPU监控 (Notebook模式)
    # 多GPU模式自动由 train_distributed.py 集成

    from unsloth_finetune.training.distributed.gpu_monitor import GPUMonitor, GPUMonitorCallback

    # 创建GPU监控器
    notebook_monitor = GPUMonitor(
        log_dir=SINGLE_GPU_LOG_DIR,
        log_interval=LOGGING_STEPS,
    )

    # 创建Callback (集成到Trainer)
    gpu_callback = GPUMonitorCallback(notebook_monitor)

    # 注意: 实际训练时会将 gpu_callback 加入 callbacks 列表:
    # trainer = SFTTrainer(
    #     model=model,
    #     train_dataset=train_dataset,
    #     args=training_args,
    #     callbacks=[gpu_callback],  # GPU监控回调
    # )

    # 训练结束后自动保存JSON汇总
    notebook_monitor.save_summary()

    print(f"\nGPU监控日志将保存到: {SINGLE_GPU_LOG_DIR}/")
    print("训练完成后查看: gpu_summary_*.json 和 gpu_log_*.csv")

⏭ 当前模式: DDP, 跳过GPU监控演示


### 10.5 性能对比框架

训练完成后，对比单GPU和多GPU的性能指标，包括训练速度、显存效率、Loss收敛一致性。

对比数据来源 (层级化路径, 通过latest.txt自动定位最新训练):

- 单GPU: `models/finetuned/gemma4_e4b_lora/single/{timestamp}/training_result.json`
- DDP 8GPU: `models/finetuned/gemma4_e4b_lora/ddp_8gpu/{timestamp}/training_result.json`
- FSDP 8GPU: `models/finetuned/gemma4_e4b_lora/fsdp_8gpu/{timestamp}/training_result.json`


In [23]:
if TRAINING_MODE != "compare":
    print(f"⏭ 当前模式: {TRAINING_MODE}, 跳过性能对比")
else:
    # 性能对比分析代码
    # ============================================================
    # 读取各模式的训练结果和GPU监控数据，生成三方对比报告
    # 数据来源优先级: training_result.json → GPU summary → 标记为N/A

    import json
    from pathlib import Path

    def load_training_result(result_path):
        path = Path(result_path)
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return None

    def load_gpu_summary(summary_dir):
        summary_dir = Path(summary_dir)
        if not summary_dir.exists():
            return None
        summary_files = sorted(summary_dir.glob("gpu_summary_*.json"))
        if not summary_files:
            return None
        with open(summary_files[-1], "r", encoding="utf-8") as f:
            return json.load(f)

    def build_mode_result(training_result, gpu_summary, mode_name):
        """从 training_result.json 和 gpu_summary 合成统一的结果字典
        training_result.json 优先; 缺失字段从 gpu_summary 补充; 仍缺失则标记 N/A"""
        result = {}
        source_labels = []

        if training_result:
            source_labels.append("training_result.json")
            result["训练时长(s)"] = training_result.get("train_runtime_sec", "N/A")
            result["最终Loss"] = training_result.get("train_loss", "N/A")
            result["吞吐量(samples/s)"] = training_result.get("samples_per_second", "N/A")
            result["峰值VRAM(GB)"] = training_result.get("peak_vram_gb", "N/A")
            result["VRAM利用率(%)"] = training_result.get("vram_utilization_pct", "N/A")
            result["GPU数量"] = training_result.get("world_size", 1)
            result["分布式模式"] = training_result.get("distributed_mode", mode_name)
            lr_key = "learning_rate_effective" if "learning_rate_effective" in training_result else "learning_rate"
            result["学习率"] = training_result.get(lr_key, "N/A")
            batch_key = "effective_global_batch_size" if "effective_global_batch_size" in training_result else "effective_batch_size"
            result["有效批次"] = training_result.get(batch_key, "N/A")
        elif gpu_summary:
            source_labels.append("gpu_summary (training_result.json缺失)")
            result["训练时长(s)"] = gpu_summary.get("total_elapsed_sec", "N/A")
            result["最终Loss"] = "N/A"
            result["吞吐量(samples/s)"] = "N/A"
            peak_vram = max(
                (s.get("max_alloc_gb", 0) for s in gpu_summary.get("per_gpu_stats", {}).values()),
                default="N/A",
            )
            result["峰值VRAM(GB)"] = peak_vram
            vram_util = [s.get("memory_efficiency_pct", 0) for s in gpu_summary.get("per_gpu_stats", {}).values()]
            result["VRAM利用率(%)"] = round(sum(vram_util) / len(vram_util), 1) if vram_util else "N/A"
            result["GPU数量"] = gpu_summary.get("gpu_count", "N/A")
            result["分布式模式"] = mode_name
            result["学习率"] = "N/A"
            result["有效批次"] = "N/A"
        else:
            source_labels.append("无数据")
            return None, source_labels

        if gpu_summary and training_result:
            peak_vram = max(
                (s.get("max_alloc_gb", 0) for s in gpu_summary.get("per_gpu_stats", {}).values()),
                default=result.get("峰值VRAM(GB)"),
            )
            if result.get("峰值VRAM(GB)", "N/A") == "N/A":
                result["峰值VRAM(GB)"] = peak_vram
            vram_util = [s.get("memory_efficiency_pct", 0) for s in gpu_summary.get("per_gpu_stats", {}).values()]
            if result.get("VRAM利用率(%)", "N/A") == "N/A":
                result["VRAM利用率(%)"] = round(sum(vram_util) / len(vram_util), 1) if vram_util else "N/A"

        return result, source_labels

    def validate_training_data_complete(training_result):
        """验证训练结果数据是否完整(不含N/A字段)

        Args:
            training_result: 训练结果字典

        Returns:
            bool: 数据是否完整
        """
        if not training_result:
            return False

        required_fields = ["train_runtime_sec", "train_loss", "samples_per_second", "peak_vram_gb", "vram_utilization_pct"]

        for field in required_fields:
            value = training_result.get(field, "N/A")
            if value == "N/A" or not isinstance(value, (int, float)):
                return False

        return True

    def calc_pairwise_comparison(base_result, target_result, base_label, target_label):
        """计算两方对比指标，自动跳过含N/A的字段"""
        comp = {}
        base_time = base_result.get("训练时长(s)", "N/A")
        target_time = target_result.get("训练时长(s)", "N/A")
        if isinstance(base_time, (int, float)) and isinstance(target_time, (int, float)) and base_time > 0 and target_time > 0:
            speedup = base_time / target_time
            comp["训练速度提升"] = f"{speedup:.2f}x"
            comp["时间节省(%)"] = f"{(1 - 1/speedup)*100:.1f}%"
        base_sps = base_result.get("吞吐量(samples/s)", "N/A")
        target_sps = target_result.get("吞吐量(samples/s)", "N/A")
        if isinstance(base_sps, (int, float)) and isinstance(target_sps, (int, float)) and base_sps > 0:
            comp["吞吐量提升"] = f"{target_sps / base_sps:.2f}x"
        base_loss = base_result.get("最终Loss", "N/A")
        target_loss = target_result.get("最终Loss", "N/A")
        if isinstance(base_loss, (int, float)) and isinstance(target_loss, (int, float)):
            comp["Loss一致性"] = f"{base_label}={base_loss:.4f}, {target_label}={target_loss:.4f}, 差异={abs(base_loss - target_loss):.4f}"
        base_vram = base_result.get("VRAM利用率(%)", "N/A")
        target_vram = target_result.get("VRAM利用率(%)", "N/A")
        if isinstance(base_vram, (int, float)) and isinstance(target_vram, (int, float)):
            comp["显存效率改善"] = f"{base_label}={base_vram:.1f}%, {target_label}={target_vram:.1f}%"
        return comp

    # 验证各模式的训练记录是否真实存在且数据完整
    print("正在验证各模式的训练记录...")

    MODE_CANDIDATES = {
        "single": {
            "label": "单GPU",
            "mode_key": "single",
            "gpu_log_dir": SINGLE_GPU_LOG_DIR,
        },
        "ddp": {
            "label": "DDP 8GPU",
            "mode_key": "DDP",
            "gpu_log_dir": DDP_GPU_LOG_DIR,
        },
        "ddp_2x": {
            "label": "DDP 8GPU + 2倍吞吐",
            "mode_key": "DDP_2X",  # 需要独立目录,不复用DDP
            "gpu_log_dir": DDP_2X_GPU_LOG_DIR,
        },
        "devicemap": {
            "label": "device_map 2D并行",
            "mode_key": "device_map",
            "gpu_log_dir": DEVICEMAP_GPU_LOG_DIR,
        },
        "fsdp": {
            "label": "FSDP 8GPU",
            "mode_key": "FSDP",
            "gpu_log_dir": FSDP_GPU_LOG_DIR,
        },
        "auto": {
            "label": "自动检测模式",
            "mode_key": "auto",
            "gpu_log_dir": AUTO_GPU_LOG_DIR,
        },
    }

    MODE_INFO = {}
    for candidate_key, candidate_info in MODE_CANDIDATES.items():
        result_dir = get_latest_output(LORA_OUTPUT_BASE, candidate_info["mode_key"])
        if not result_dir:
            print(f"⚠️ {candidate_info['label']}: 无训练记录(latest.txt不存在)")
            continue

        tr_path = Path(result_dir) / "training_result.json"
        if not tr_path.exists():
            print(f"⚠️ {candidate_info['label']}: training_result.json不存在")
            continue

        tr_data = load_training_result(str(tr_path))
        if not validate_training_data_complete(tr_data):
            print(f"⚠️ {candidate_info['label']}: 训练数据不完整(含N/A字段)")
            continue

        MODE_INFO[candidate_key] = {
            "label": candidate_info["label"],
            "result_dir": result_dir,
            "gpu_log_dir": candidate_info["gpu_log_dir"],
            "tr_path": str(tr_path),
        }
        print(f"✅ {candidate_info['label']}: 数据验证通过")

    print(f"\n检测到 {len(MODE_INFO)} 个有效的训练记录")
    if len(MODE_INFO) == 0:
        print("尚未完成任何有效的训练，暂无对比数据")
        print("请先完成至少一种模式的训练，再运行此对比分析")
        print("\n预期对比结果 (基于8×A6000配置):")
        print("  训练速度提升: ~6.8x")
        print("  吞吐量提升: ~6.8x")
        print("  时间节省: ~85%")
        print("  Loss一致性: 差异<5%")
        print("  显存利用率: 从~21%提升至~74%")
    else:
        mode_results = {}
        mode_sources = {}
        for mode_key, info in MODE_INFO.items():
            tr_data = load_training_result(info["tr_path"])
            gs_data = load_gpu_summary(info["gpu_log_dir"])
            built, sources = build_mode_result(tr_data, gs_data, info["label"])
            if built:
                mode_results[mode_key] = built
                mode_sources[mode_key] = sources
            print(f"{info['label']}结果路径: {info['tr_path']}")
            print(f"{info['label']}数据来源: {', '.join(sources)}")

        # 生成对比报告
        report = {"各模式配置": {}, "对比结果": {}}
        for mode_key, result in mode_results.items():
            label = MODE_INFO[mode_key]["label"]
            report["各模式配置"][label] = result

        # 生成两方对比 (以单GPU为基准, 若单GPU无数据则用DDP)
        available_modes = list(mode_results.keys())
        if len(available_modes) >= 2:
            base_mode = "single" if "single" in available_modes else available_modes[0]
            base_label = MODE_INFO[base_mode]["label"]
            for target_key in available_modes:
                if target_key == base_mode:
                    continue
                target_label = MODE_INFO[target_key]["label"]
                comp = calc_pairwise_comparison(mode_results[base_mode], mode_results[target_key], base_label, target_label)
                report["对比结果"][f"{base_label} vs {target_label}"] = comp

            # 动态生成多GPU模式之间的对比(以DDP为基准,如果存在)
            multi_gpu_modes = ["ddp", "ddp_2x", "devicemap", "fsdp", "auto"]
            if "ddp" in available_modes:
                ddp_label = MODE_INFO["ddp"]["label"]
                for target_key in available_modes:
                    if target_key in multi_gpu_modes and target_key != "ddp":
                        target_label = MODE_INFO[target_key]["label"]
                        comp = calc_pairwise_comparison(
                            mode_results["ddp"], mode_results[target_key], ddp_label, target_label
                        )
                        report["对比结果"][f"{ddp_label} vs {target_label}"] = comp

        if mode_results:
            print("\n" + "=" * 60)
            print("性能对比报告")
            print("=" * 60)

            for section, data in report.items():
                if data:
                    print(f"\n{section}:")
                    for key, value in data.items():
                        if isinstance(value, dict):
                            print(f"\n  {key}:")
                            for k, v in value.items():
                                print(f"    {k}: {v}")
                        else:
                            print(f"  {key}: {value}")

            print("\n" + "=" * 60)

            # 保存对比报告
            report_path = Path("benchmark_results/comparison_report.json")
            report_path.parent.mkdir(parents=True, exist_ok=True)
            with open(report_path, "w", encoding="utf-8") as f:
                json.dump(report, f, indent=2, ensure_ascii=False)
            print(f"对比报告已保存: {report_path}")

            # 数据来源说明
            print("\n数据来源说明:")
            for mode_key, sources in mode_sources.items():
                label = MODE_INFO[mode_key]["label"]
                print(f"  {label}: {', '.join(sources)}")
            if any("gpu_summary" in s for sources in mode_sources.values() for s in sources):
                print("\n⚠️ 部分数据来自GPU监控日志(非训练结果文件), Loss和吞吐量字段可能为N/A")
                print("  建议重新运行TRAINING_MODE='single'以生成完整的training_result.json")
        else:
            print("\n尚未完成任何训练，暂无对比数据")
            print("请先完成至少两种模式的训练，再运行此对比分析")
            print("\n预期对比结果 (基于8×A6000配置):")
            print("  训练速度提升: ~6.8x")
            print("  吞吐量提升: ~6.8x")
            print("  时间节省: ~85%")
            print("  Loss一致性: 差异<5%")
            print("  显存利用率: 从~21%提升至~74%")

⏭ 当前模式: DDP, 跳过性能对比


---

## 总结

### 8×A6000 多GPU优化效果

| 指标 | 单GPU | 8×A6000 DDP | 提升比例 |
|------|-------|-------------|----------|
| 训练速度 | 基准 | ~6.8x加速 | +680% |
| 显存利用率 | ~21% | ~74% | +250% |
| 有效批次大小 | 8 | 64 | +700% |
| 学习率 | 2e-5 | 3.2e-4 (线性缩放) | 自适应 |
| Loss收敛 | 基准 | ≈基准 (差异<5%) | 等效 |

### 单GPU vs 多GPU训练选择指南

| 场景 | 推荐方式 |
|------|----------|
| E4B模型，单卡VRAM >= 10GB | 单GPU训练（本Notebook前9节） |
| E4B模型，8×A6000服务器 | **DDP 8卡训练** (本节优化) |
| 31B+大模型，单卡VRAM不足 | FSDP分片训练 |
| 多机多卡，大规模训练 | DDP/FSDP + 多机 |

### 关键优化要点
1. **DDP > FSDP** (对于E4B): 模型可放入单卡，DDP通信开销更低
2. **BF16混合精度**: A6000 Ampere架构原生支持，计算效率提升约2x
3. **batch_size=4**: 48GB VRAM下QLoRA E4B仅需~10GB，增大批次可提升吞吐
4. **线性LR缩放**: `lr = base_lr × world_size` 确保收敛稳定
5. **GPU监控**: 实时追踪显存/利用率，避免OOM并优化负载均衡

### 部署建议
1. **本地部署**: 使用GGUF格式 + Ollama
2. **API服务**: 使用vLLM加载LoRA adapters
3. **云端部署**: 推送到Hugging Face Hub

---

**参考资源：**

- [Unsloth多GPU文档](https://unsloth.ai/docs/basics/multi-gpu-training-with-unsloth)
- [Unsloth官方文档](https://unsloth.ai/)
- [Gemma 4模型](https://huggingface.co/google/gemma-4)
- [TRL库](https://github.com/huggingface/trl)
- [PyTorch DDP文档](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html)
- [PyTorch FSDP文档](https://pytorch.org/tutorials/intermediate/FSDP_tutorial.html)
